# Benchmark construction inverts the measured value of LLM re-ranking in rare disease diagnosis

**v5 — analysis notebook for the AIIM/JBI submission.**

## What this paper claims

1. **The inversion.** A phenotype benchmark built by sampling HPO terms from the same
   Orphanet annotations that the retrieval index is built from makes lexical/ontological
   retrieval look near-perfect (R@5 up to 0.97) and makes LLM re-ranking look actively
   *harmful* (delta up to -0.24). On real patient phenotype sets (RareBench) the sign
   **reverses**: retrieval collapses and LLM re-ranking gives a large positive delta.
2. **Tuning transfer failure.** Fusion weights selected on a held-out *dev split of the
   same synthetic construction* (`hybrid_tuned`) are the best internal retriever and one of
   the **worst** external ones. Tuning on same-source data does not merely fail to help —
   it actively degrades external performance.
3. **Cause.** Not a pipeline bug. Synthetic-from-gold controls recover the true disease at
   internal rates on the *same* diseases; the gap is a property of the query term
   distribution (real cases use terms Orphanet never annotates: onset/death terms,
   lab-derived terms).

The RAG system is the **instrument**, not the contribution. Every component (BM25, RRF,
IC overlap, LLM re-ranking) is off the shelf. Do not write this up as a system paper.

## What changed from v4

| # | change | why |
|---|---|---|
| 1 | External LLM re-rank runs on **all 834 mapped cases**, not a 120-case sample | v4's headline claim rested on n=67 conditional. Not defensible. |
| 2 | Gemini **dropped from the external arm** (kept internally, where it is complete at 400/400) | v4's external Gemini covered 30/120 because of the 500 RPD cap. A member covering 25% of cases cannot be in an ensemble. |
| 3 | Ensemble demoted to a **secondary, negative result** | Fusion (0.821) lost to MedGemma alone (0.866). Report it as no-benefit-at-this-n, do not sell it. |
| 4 | External LLM arm uses `EXT_RETRIEVER` (default `hybrid`), not `hybrid_tuned` | v4 fed the LLMs the retriever with the *lowest* external ceiling (0.493 vs 0.607). See the honesty note in the config cell. |
| 5 | **Three phenotype-driven baselines added**: Phrank-style, LIRICAL-style LR, Resnik/Phenomizer | A reviewer will not accept a rare-disease ranking paper with no comparison to the standard prioritisers. |
| 6 | **RareBench published GPT-4 numbers** included, with an explicit non-comparability block | We use their data; we must sit next to their reported results — and be honest that the tasks differ. |
| 7 | **Holm + Benjamini-Hochberg correction** across pre-declared test families | v4 ran dozens of McNemar tests uncorrected. |
| 8 | HPO release **pinned and recorded** | v4 pulled `master`. Non-reproducible, and RareBench used 2023-06-06. |
| 9 | **Batched local generation** with OOM backoff | 834 sequential MedGemma calls is ~10 h. Batching is what makes this fit a session. |
| 10 | Interactive Flask/ngrok demo **deleted** | Unevaluated, and it emitted a wrong gene attribution. Either evaluate it with a clinician or cut it. It is cut. |
| 11 | Final tables **recomputed from artefacts**, no hardcoded constants | v4's summary tables hardcoded numbers copied from earlier prints. |

## Honest limitations to carry into the paper

* The three baselines are **re-implementations from published descriptions**, not the
  authors' code. They must be labelled as such. See the baseline cell for exact formulas.
* MedGemma results are **4-bit NF4 on Turing**. Quantisation is a confound; state it.
* 288 of 1,122 RareBench cases are dropped (no ORPHA code / not in KB). LIRICAL keeps only
  53%. The coverage table is a required limitation, not a footnote.
* This notebook contains no clinical evaluation. AIIM strongly recommends one. If you
  submit there, add a clinician-read error taxonomy or expect that objection.


## K0. Environment, paths, config

In [ ]:
# =====================================================================
# K0. Environment detection, paths, config
# =====================================================================
import os, re, sys, json, math, time, glob, pickle, random, warnings
from collections import defaultdict
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=UserWarning)

SEED = 42
random.seed(SEED); np.random.seed(SEED)

ON_KAGGLE = os.path.isdir("/kaggle/input")
ON_COLAB  = "google.colab" in sys.modules

if ON_KAGGLE:
    OUT = "/kaggle/working/"
    # keep the big HF cache off the 20 GB working dir
    os.environ.setdefault("HF_HOME", "/kaggle/temp/hf")
    hits = glob.glob("/kaggle/input/**/en_product4.csv", recursive=True)
    if not hits:
        hits = [p for p in glob.glob("/kaggle/input/**/*.csv", recursive=True)
                if "product4" in os.path.basename(p).lower()]
    assert hits, ("en_product4.csv not found under /kaggle/input. "
                  "Attach it as a Dataset (+ Add Input) and re-run this cell.")
    RAW_CSV = hits[0]
elif ON_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT = "/content/drive/MyDrive/Thesis_dataset_knowledgebase/"
    RAW_CSV = OUT + "en_product4.csv"
else:
    OUT = "./"
    RAW_CSV = "./en_product4.csv"

os.makedirs(OUT, exist_ok=True)

# ---- experiment knobs -------------------------------------------------
N_TEST         = 100    # cases per tier -> 100 x 4 = 400 evaluations per model
N_DEV          = 60     # held-out diseases for tuning ONLY. Never scored.
TOP_K_RETRIEVE = 25     # candidates handed to each LLM
TIERS          = ["T0", "T1", "T2", "T3"]

# ---- MedGemma source --------------------------------------------------
# Prefer a Kaggle Models mount (no download). Leave None to pull from HF.
MEDGEMMA_PATH = None                              # e.g. "/kaggle/input/medgemma/transformers/27b-text-it/1"
MEDGEMMA_ID   = "google/medgemma-27b-text-it"

print(f"platform : {'kaggle' if ON_KAGGLE else 'colab' if ON_COLAB else 'local'}")
print(f"RAW_CSV  : {RAW_CSV}")
print(f"OUT      : {OUT}")
print(f"workload : {N_TEST} x {len(TIERS)} = {N_TEST*len(TIERS)} evals per model")

try:
    import torch
    print(f"torch    : {torch.__version__} | cuda={torch.cuda.is_available()} "
          f"| n_gpu={torch.cuda.device_count()}")
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        print(f"  gpu{i}: {p.name} {p.total_memory/2**30:.1f} GiB sm_{p.major}{p.minor}")
    if torch.cuda.device_count() < 2:
        print("\n[!] Fewer than 2 GPUs visible. MedGemma 27B at 4-bit needs ~15-16 GiB;")
        print("    one T4 gives ~15 GiB usable, which is too tight with KV cache.")
        print("    Settings -> Accelerator -> GPU T4 x2.")
except ImportError:
    print("torch not importable yet — the install cell will handle it.")

In [ ]:
# =====================================================================
# K0c. v5 configuration. Everything the reframing depends on lives here.
# =====================================================================

# ---- ensembles -------------------------------------------------------
# Internal: v4's checkpoint already holds 400/400 for all three members, so
# there is no reason to discard Gemini internally. External: Gemini's 500 RPD
# cap makes 834 calls a two-day run, so it is OFF by default and the external
# ensemble is the two lineage-diverse members.
ENSEMBLE_INTERNAL = ["google/medgemma-27b-text-it",
                     "gemini-3.5-flash-lite",
                     "mistral-small-latest"]
EXT_INCLUDE_GEMINI = False          # True only if you will spend 2+ days on it
ENSEMBLE_EXTERNAL  = ["google/medgemma-27b-text-it", "mistral-small-latest"]
if EXT_INCLUDE_GEMINI:
    ENSEMBLE_EXTERNAL.insert(1, "gemini-3.5-flash-lite")

# ---- which retriever feeds the external LLM stage --------------------
#
#   HONESTY NOTE — WRITE THIS INTO THE PAPER, DO NOT BURY IT.
#
#   v4 fed the external LLM stage `hybrid_tuned` (R@25 = 0.493 externally) on the
#   grounds that it was pre-specified on the dev split. That is defensible as
#   pre-registration but it ceiling-limits the very measurement the paper is about:
#   a re-ranker cannot recover a disease that was never retrieved.
#
#   v5 uses `hybrid` (equal-weight RRF, R@25 = 0.607 externally). This is a
#   POST-HOC, EXTERNALLY-INFORMED choice. Declare it as such. The defence is:
#     (a) the tuning-transfer result is retrieval-only and needs no LLM, so it is
#         unaffected by this choice and is reported separately;
#     (b) the primary inversion analysis is reported CONDITIONAL on the gold being
#         retrievable, which measures re-ranking and is the quantity of interest;
#     (c) the unconditional numbers are reported too, for both retrievers on the
#         retrieval side, so a reader can see exactly what the choice bought.
#
#   Set EXT_RETRIEVER = "hybrid_tuned" to reproduce v4's pre-specified arm.
EXT_RETRIEVER = "hybrid"

# ---- local generation ------------------------------------------------
LOCAL_BATCH          = 4      # starting batch size; halves automatically on OOM
LOCAL_MAX_NEW_TOKENS = 200    # 25 integers + JSON wrapper is ~110 tokens
USE_FP16_COMPUTE     = True   # v4 used fp32 compute on T4, which forgoes tensor
                              # cores. The load cell probes fp16 for finite logits
                              # and falls back to fp32 if the probe fails.

# ---- session budget --------------------------------------------------
# Kaggle documents 12 h per notebook session and roughly 30 h/week of accelerator
# time. VERIFY BOTH AGAINST YOUR OWN ACCOUNT — these change.
SESSION_BUDGET_H = 10.5       # runners stop cleanly before the session is killed
                              # and the checkpoint resumes on the next session.

# ---- HPO version -----------------------------------------------------
# v4 pulled master, which is not reproducible. RareBench used hp 2023-06-06.
# We try the pin, fall back to master, and RECORD whichever was used.
HPO_PINNED_TAG = "v2023-06-06"

# ---- statistics ------------------------------------------------------
ALPHA        = 0.05
CORRECTIONS  = ("holm", "fdr_bh")

print("ENSEMBLE_INTERNAL:", [m.split('/')[-1] for m in ENSEMBLE_INTERNAL])
print("ENSEMBLE_EXTERNAL:", [m.split('/')[-1] for m in ENSEMBLE_EXTERNAL])
print("EXT_RETRIEVER    :", EXT_RETRIEVER, " (post-hoc choice — declare it)")
print("LOCAL_BATCH      :", LOCAL_BATCH, "| max_new_tokens", LOCAL_MAX_NEW_TOKENS)
print("session budget   :", SESSION_BUDGET_H, "h")


## K0e. Restore cached artefacts from your saved v4 outputs

**This is what lets one session be enough.** The internal B4 re-rank (400 cases x 3
models) is unchanged from v4 and is the single biggest cost in the whole notebook
(~5 h). If you still have v4's output files, attach them as a Kaggle input dataset
and this cell restores them before anything recomputes.

**How to attach them:**
1. On the v4 notebook (or wherever you downloaded its outputs), get the files onto
   your machine — `Output` tab -> download, or you may already have them locally.
2. On kaggle.com -> **Datasets -> New Dataset** -> drag in the files (at minimum
   `llm_rerank_checkpoint.pkl`; the others below are optional extras).
3. In *this* notebook's editor: **+ Add Input** -> search your new dataset -> attach.
4. Run this cell. It searches every attached input for filename matches and copies
   them into `/kaggle/working/`, where every downstream cell already checks
   `os.path.exists(...)` before doing the corresponding work — restoring the file
   is the only change needed; no other cell needs editing.

If you'd rather point at a specific path than auto-search, set `RESTORE_FROM`
below to it (e.g. `"/kaggle/input/my-v4-outputs"`).


In [ ]:
# =====================================================================
# K0e. Restore cached artefacts from a previously attached Kaggle dataset.
# =====================================================================
import shutil

RESTORE_FROM = None   # e.g. "/kaggle/input/my-v4-outputs"; None = auto-search all inputs

# filename -> what restoring it skips. Every one of these is checked with
# os.path.exists() by a later cell before it does the corresponding work, so
# restoring the file is sufficient — nothing else needs to change.
RESTORE_MANIFEST = {
    "rarebench_rerank_v5.pkl":   "resumes the external 834-case run from 344 done",
    "llm_rerank_checkpoint.pkl": "skips the internal B4 re-rank, 400x3 calls, ~5 h",
    "hp.obo":                    "skips an 11 MB re-download",
    "emb_sapbert_withname.npy":  "skips a SapBERT encoding pass over the KB",
    "emb_sapbert_noname.npy":    "skips a SapBERT encoding pass over the KB",
    "emb_minilm_withname.npy":   "skips a MiniLM encoding pass over the KB",
    "emb_minilm_noname.npy":     "skips a MiniLM encoding pass over the KB",
}
def _restore_roots():
    return [RESTORE_FROM] if RESTORE_FROM else glob.glob("/kaggle/input/*")

roots = _restore_roots()
restored, missing = [], []
for fname, benefit in RESTORE_MANIFEST.items():
    dst = OUT + fname
    if os.path.exists(dst):
        print(f"  already present   {fname:28s} (this session)")
        continue
    hit = next((p for r in roots
               for p in glob.glob(os.path.join(r, "**", fname), recursive=True)), None)
    if hit:
        shutil.copy2(hit, dst)
        restored.append(fname)
        print(f"  restored          {fname:28s} <- {hit}")
        print(f"                    {'':28s}    {benefit}")
    else:
        missing.append(fname)

print(f"\n{len(restored)} restored, {len(missing)} not found "
      f"(computed fresh): {missing or 'none'}")
if "llm_rerank_checkpoint.pkl" in restored:
    print("\n>>> Internal B4 checkpoint restored — B4 will report near-zero remaining")
    print(">>> work when you reach it. This is the ~5 h saving.")
elif "llm_rerank_checkpoint.pkl" in missing:
    print("\n>>> No internal checkpoint found. B4 will run the full 400x3 internal")
    print(">>> re-rank (~5 h). If you have v4's outputs, attach them (see markdown")
    print(">>> above) and re-run this cell before B4.")

# ---- optional: reuse v4's partial EXTERNAL re-rank (120 RareBench cases) -----
# v4's external checkpoint (`rarebench_rerank_checkpoint.pkl`) was built with
# candidates from `hybrid_tuned`. Its (case_id, model) keys are byte-identical
# in shape to v5's, so it can be reused DIRECTLY as a warm start for R4 — but
# ONLY if EXT_RETRIEVER == "hybrid_tuned" (see K0c). Reusing it under the
# default EXT_RETRIEVER = "hybrid" would silently mix candidate sets across
# cases and bias the result — do not do that.
_old_ext = next((p for r in roots
                 for p in glob.glob(os.path.join(r, "**",
                     "rarebench_rerank_checkpoint.pkl"), recursive=True)), None)
if _old_ext:
    print(f"\nFound v4's external sample checkpoint: {_old_ext}")
    if EXT_RETRIEVER == "hybrid_tuned":
        dst = OUT + "rarebench_rerank_v5.pkl"
        if not os.path.exists(dst):
            shutil.copy2(_old_ext, dst)
            print(">>> EXT_RETRIEVER == 'hybrid_tuned' — copied as a warm start for R4.")
            print(">>> R4's completeness audit will show ~120 cases already done")
            print(">>> (up to ~1.5 h saved on MedGemma) and will fetch the rest.")
    else:
        print(f">>> EXT_RETRIEVER is '{EXT_RETRIEVER}', not 'hybrid_tuned' — NOT reused")
        print(">>> (candidates would not match; reusing it would bias the result).")
        print(">>> Set EXT_RETRIEVER = 'hybrid_tuned' in K0c if you want this reuse;")
        print(">>> that also reverts to v4's pre-registered retriever choice.")

# ---- verify the restored external checkpoint matches this session ----------
_p = OUT + "rarebench_rerank_v5.pkl"
if os.path.exists(_p):
    with open(_p, "rb") as f:
        _s = pickle.load(f)
    _was = _s.get(("meta",), {}).get("retriever")
    _done = {m: sum(1 for k, v in _s.items()
                    if isinstance(k, tuple) and len(k) == 2 and k[1] == m
                    and isinstance(v, dict) and v.get("status") == "ok")
             for m in ENSEMBLE_EXTERNAL}
    print(f"\nexternal checkpoint: built with retriever={_was!r}, "
          f"session uses {EXT_RETRIEVER!r}")
    print(f"already completed: {_done}")
    assert _was == EXT_RETRIEVER, (
        f"RETRIEVER MISMATCH ({_was!r} vs {EXT_RETRIEVER!r}). Delete the restored "
        "checkpoint or change EXT_RETRIEVER in K0c. Mixing candidate sets across "
        "cases would silently bias the headline result.")
    print(">>> R4 will resume from here and re-pay for nothing.")

In [ ]:
# =====================================================================
# K0b. Installs. Needs internet ON (Settings -> Turn on internet).
# =====================================================================
!pip install -q -U sentence-transformers accelerate bitsandbytes transformers
!pip install -q -U google-genai mistralai
!pip install -q -U statsmodels

import importlib, sentence_transformers, transformers
print("sentence-transformers", sentence_transformers.__version__)
print("transformers        ", transformers.__version__)
try:
    import bitsandbytes as bnb; print("bitsandbytes        ", bnb.__version__)
except Exception as e:
    print("bitsandbytes        FAILED:", e)
try:
    import accelerate; print("accelerate          ", accelerate.__version__)
except Exception as e:
    print("accelerate          FAILED:", e)

print("\n>>> Record these versions in your Methods section. A 4-bit result is")
print(">>> version-sensitive and not reproducible without them.")

In [ ]:
# =====================================================================
# S0. p-value registry. Every test in this notebook registers here, and
#     the correction cell at the end applies Holm + BH within families.
#     A test that is not registered does not appear in the paper.
# =====================================================================
from statsmodels.stats.multitest import multipletests

PVALS = []

def reg_p(family, label, p, **extra):
    """Record a raw p-value under a pre-declared test family."""
    PVALS.append({"family": family, "label": label, "p_raw": float(p), **extra})
    return float(p)

def pval_table(alpha=ALPHA, methods=CORRECTIONS):
    if not PVALS:
        return pd.DataFrame()
    df = pd.DataFrame(PVALS)
    for meth in methods:
        col_p, col_r = f"p_{meth}", f"sig_{meth}"
        df[col_p] = np.nan
        df[col_r] = False
        for fam, g in df.groupby("family"):
            rej, padj, _, _ = multipletests(g["p_raw"].values, alpha=alpha, method=meth)
            df.loc[g.index, col_p] = padj
            df.loc[g.index, col_r] = rej
    return df

# Pre-declared families. Declaring them BEFORE running the tests is the point;
# choosing family boundaries after seeing p-values is p-hacking by another name.
FAMILIES = {
    "F1_internal_retrieval":  "tuned vs equal-weight vs BM25 on the internal test set",
    "F2_internal_llm":        "each LLM system vs its retrieval baseline, per tier",
    "F3_external_retrieval":  "retriever and baseline comparisons on RareBench",
    "F4_external_llm":        "LLM systems vs retrieval on RareBench",
}
for k, v in FAMILIES.items():
    print(f"  {k:26s} {v}")
print(f"\nRegistry armed. alpha={ALPHA}, corrections={CORRECTIONS}")


## A1. Knowledge base from Orphanet product4

In [ ]:
# =====================================================================
# A1. Rebuild KB from raw Orphanet product4, carrying HPO IDs
# =====================================================================
raw = pd.read_csv(RAW_CSV, low_memory=False)
print(f"Loaded {len(raw):,} rows")

COLS = dict(orpha="OrphaCode", name="DisorderName",
            hpo_id="HPO_ID", hpo_term="HPO_Term", freq="HPO_Frequency")
missing = [c for c in COLS.values() if c not in raw.columns]
assert not missing, f"Missing expected columns: {missing}. Found: {list(raw.columns)}"

df = pd.DataFrame({
    "orpha_id":      raw[COLS["orpha"]].astype(str),
    "disease_name":  raw[COLS["name"]].astype(str),
    "hpo_id":        raw[COLS["hpo_id"]].astype(str),
    "hpo_term":      raw[COLS["hpo_term"]].astype(str),
    "frequency_raw": raw[COLS["freq"]].astype(str),
})
df = df.dropna(subset=["disease_name", "hpo_term"])
df = df[(df["hpo_term"].str.strip() != "") & (df["hpo_term"].str.lower() != "nan")]

FREQUENCY_MAP = {
    "obligate": 1.0, "always present": 1.0,
    "very frequent": 0.9,
    "frequent": 0.6,
    "occasional": 0.3,
    "very rare": 0.1, "rare": 0.1,
    "excluded": 0.0,
}
def freq_to_weight(f):
    fl = str(f).strip().lower()
    if fl in ("nan", "none", ""):
        return 0.5
    for key in ("obligate", "always present", "very frequent", "very rare",
                "frequent", "occasional", "excluded", "rare"):
        if key in fl:
            return FREQUENCY_MAP[key]
    return 0.5

df["freq_weight"] = df["frequency_raw"].apply(freq_to_weight)
n0 = len(df)
df = df[df["freq_weight"] > 0.0]
print(f"Dropped {n0 - len(df):,} excluded/negated associations")

records = []
for (oid, name), g in df.groupby(["orpha_id", "disease_name"], sort=True):
    g = g.sort_values("freq_weight", ascending=False)
    if len(g) < 3:
        continue
    records.append({
        "orpha_id": oid, "disease_name": name,
        "hpo_ids": g["hpo_id"].tolist(),
        "symptoms": g["hpo_term"].tolist(),
        "symptom_weights": g["freq_weight"].tolist(),
        "n_symptoms": len(g),
    })

kb = pd.DataFrame(records).reset_index(drop=True)
kb["text_with_name"] = kb.apply(
    lambda r: f"{r['disease_name']}. Symptoms: " + ", ".join(r["symptoms"]), axis=1)
kb["text_no_name"] = kb["symptoms"].apply(lambda s: "Symptoms: " + ", ".join(s))

kb.to_pickle(OUT + "disease_kb_v3.pkl")
print(f"KB: {len(kb):,} diseases | median symptoms {kb.n_symptoms.median():.0f}")

## A2. HPO ontology: ancestor closure + information content

The release is now **pinned and recorded**. v4 pulled `master`, which makes the IC values, the T3 parent-generalisation tier and the ancestor closure irreproducible. RareBench used hp 2023-06-06.

In [ ]:
# =====================================================================
# A2. Parse hp.obo, build ancestor closure, compute IC
# =====================================================================
HPO_OBO = OUT + "hp.obo"
HPO_URL_PINNED = ("https://raw.githubusercontent.com/obophenotype/"
                  f"human-phenotype-ontology/{HPO_PINNED_TAG}/hp.obo")
HPO_URL_MASTER = ("https://raw.githubusercontent.com/obophenotype/"
                  "human-phenotype-ontology/master/hp.obo")
HPO_URL = HPO_URL_PINNED

if not os.path.exists(HPO_OBO):
    print(f"Downloading hp.obo at pinned tag {HPO_PINNED_TAG} ...")
    !curl -sfL -o "{HPO_OBO}" "{HPO_URL}" --max-time 300
    if (not os.path.exists(HPO_OBO)) or os.path.getsize(HPO_OBO) < 1_000_000:
        print(f"[!] Pinned tag {HPO_PINNED_TAG} not fetchable. Falling back to master.")
        print("[!] RECORD THIS IN THE PAPER — the run is then not version-pinned.")
        HPO_URL = HPO_URL_MASTER
        !curl -sfL -o "{HPO_OBO}" "{HPO_URL}" --max-time 300
print("hp.obo size:", os.path.getsize(HPO_OBO) if os.path.exists(HPO_OBO) else "MISSING")
print("hp.obo source:", HPO_URL)

HPO_DATA_VERSION = "unknown"
with open(HPO_OBO, encoding="utf-8") as _f:
    for _l in _f:
        if _l.startswith("data-version:"):
            HPO_DATA_VERSION = _l.split(":", 1)[1].strip(); break
        if _l.startswith("[Term]"):
            break
print("hp.obo data-version:", HPO_DATA_VERSION, " <-- QUOTE THIS IN METHODS")


def parse_hpo_obo(path):
    """Minimal OBO parser: id, name, is_a parents, alt_ids. Skips obsolete terms."""
    terms, alt_to_main, cur = {}, {}, None
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.rstrip("\n")
            if line.startswith("[") and line.endswith("]"):
                if cur and cur["id"] and not cur["obsolete"]:
                    terms[cur["id"]] = cur
                cur = ({"id": None, "name": None, "parents": [], "alts": [], "obsolete": False}
                       if line == "[Term]" else None)
                continue
            if cur is None:
                continue
            if line.startswith("id: HP:"):
                cur["id"] = line[4:].strip()
            elif line.startswith("name: "):
                cur["name"] = line[6:].strip()
            elif line.startswith("is_a: HP:"):
                cur["parents"].append(line[6:].split("!")[0].strip())
            elif line.startswith("alt_id: HP:"):
                cur["alts"].append(line[8:].strip())
            elif line.startswith("is_obsolete: true"):
                cur["obsolete"] = True
    if cur and cur["id"] and not cur["obsolete"]:
        terms[cur["id"]] = cur
    for tid, t in terms.items():
        for a in t["alts"]:
            alt_to_main[a] = tid
    return terms, alt_to_main


HAVE_ONTOLOGY = False
try:
    HPO, ALT2MAIN = parse_hpo_obo(HPO_OBO)
    assert len(HPO) > 5000
    HAVE_ONTOLOGY = True
    print(f"Parsed {len(HPO):,} HPO terms, {len(ALT2MAIN):,} alt-ids")
except Exception as e:
    HPO, ALT2MAIN = {}, {}
    print(f"[WARN] Ontology unavailable ({e}). T3 tier and IC scorer will be degraded.")


def canon(hid):
    return ALT2MAIN.get(hid, hid)


def build_ancestors(hpo):
    parents = {t: hpo[t]["parents"] for t in hpo}
    anc, WHITE, GREY = {}, 0, 1
    state = {}
    def rec(t):
        if t in anc:
            return anc[t]
        if state.get(t) == GREY:
            return {t}
        state[t] = GREY
        s = {t}
        for p in parents.get(t, []):
            if p in parents:
                s |= rec(p)
        state[t] = WHITE
        anc[t] = s
        return s
    for t in list(parents):
        rec(t)
    return anc


ANC = build_ancestors(HPO) if HAVE_ONTOLOGY else {}

def ancestors_of(hid):
    hid = canon(hid)
    return ANC.get(hid, {hid})

N_DIS = len(kb)
term_doc_count = defaultdict(int)
for ids in kb["hpo_ids"]:
    prop = set()
    for h in ids:
        prop |= ancestors_of(h)
    for t in prop:
        term_doc_count[t] += 1

IC = {t: -math.log(c / N_DIS) for t, c in term_doc_count.items() if c > 0}
MAX_IC = max(IC.values()) if IC else 1.0
print(f"IC computed for {len(IC):,} terms (max IC = {MAX_IC:.2f})")

## A3. Noise-tiered test set + held-out dev split

In [ ]:
# =====================================================================
# A3. Build tiered sets for TEST (reported) and DEV (tuning only)
# =====================================================================
DROP_KEEP_FRAC = 0.55   # T1/T3: fraction of true terms retained
DISTRACT_FRAC  = 0.40   # T2/T3: distractors as a fraction of retained terms
GENERALISE_P   = 0.50   # T3: probability a term is replaced by its parent


def sample_true_terms(row, rng_np):
    """T0: frequency-weighted sample of 4-8 of the disease's own annotations."""
    ids, names, w = row["hpo_ids"], row["symptoms"], np.array(row["symptom_weights"])
    n = min(len(ids), rng_np.randint(4, 9))
    p = (w + 0.05) / (w + 0.05).sum()
    sel = rng_np.choice(len(ids), size=n, replace=False, p=p)
    return [ids[i] for i in sel], [names[i] for i in sel]


def apply_tier(ids, names, tier, rng_py, rng_np):
    ids, names = list(ids), list(names)

    if tier in ("T1", "T3"):
        keep = max(2, int(round(len(ids) * DROP_KEEP_FRAC)))
        sel = rng_py.sample(range(len(ids)), keep)
        ids = [ids[i] for i in sel]; names = [names[i] for i in sel]

    if tier in ("T2", "T3"):
        n_d = max(1, int(round(len(ids) * DISTRACT_FRAC)))
        for _ in range(n_d):
            j = rng_np.randint(len(kb))
            other = kb.iloc[j]
            k = rng_py.randrange(len(other["hpo_ids"]))
            ids.append(other["hpo_ids"][k]); names.append(other["symptoms"][k])

    if tier == "T3" and HAVE_ONTOLOGY:
        for i in range(len(ids)):
            if rng_py.random() < GENERALISE_P:
                ps = [p for p in HPO.get(canon(ids[i]), {}).get("parents", [])
                      if p in HPO and HPO[p]["name"] not in ("All", "Phenotypic abnormality")]
                if ps:
                    p = rng_py.choice(ps)
                    ids[i], names[i] = p, HPO[p]["name"]

    order = list(range(len(ids)))
    rng_py.shuffle(order)                       # no positional leakage
    return [ids[i] for i in order], [names[i] for i in order]


def build_tiered_sets(base_cases, tiers=TIERS, seed=SEED):
    """RNGs are re-seeded per tier, exactly as in v3, so tiers stay paired."""
    out = {}
    for tier in tiers:
        rng_t = random.Random(seed)
        np_t  = np.random.RandomState(seed)
        rows = []
        for _, row in base_cases.iterrows():
            t_ids, t_names = sample_true_terms(row, np_t)
            if tier != "T0":
                t_ids, t_names = apply_tier(t_ids, t_names, tier, rng_t, np_t)
            rows.append({
                "case_id":        f"{row['orpha_id']}",
                "orpha_id":       row["orpha_id"],
                "true_disease":   row["disease_name"],
                "query_hpo_ids":  t_ids,
                "query_symptoms": t_names,
                "query_text":     "Patient presents with: " + ", ".join(t_names),
            })
        out[tier] = pd.DataFrame(rows)
    return out


# ---- TEST: identical draw to v3 ---------------------------------------
_np_test = np.random.RandomState(SEED)
test_idx = _np_test.choice(len(kb), size=min(N_TEST, len(kb)), replace=False)
test_sets = build_tiered_sets(kb.iloc[test_idx].reset_index(drop=True), TIERS, SEED)

# ---- DEV: disjoint diseases, different noise seed ---------------------
_pool   = np.setdiff1d(np.arange(len(kb)), test_idx)
_np_dev = np.random.RandomState(SEED + 1)
dev_idx = _np_dev.choice(_pool, size=min(N_DEV, len(_pool)), replace=False)
dev_sets = build_tiered_sets(kb.iloc[dev_idx].reset_index(drop=True), TIERS, SEED + 1)

overlap = set(test_idx.tolist()) & set(dev_idx.tolist())
assert not overlap, f"dev/test disease overlap: {len(overlap)}"
print(f"TEST diseases: {len(test_idx)}   DEV diseases: {len(dev_idx)}   overlap: 0")

with open(OUT + "test_sets_tiered.pkl", "wb") as f:
    pickle.dump(test_sets, f)
with open(OUT + "dev_sets_tiered.pkl", "wb") as f:
    pickle.dump(dev_sets, f)

print(f"\n{'tier':6s}{'test n':>8s}{'test terms':>12s}{'dev n':>8s}{'dev terms':>12s}")
for t in TIERS:
    print(f"{t:6s}{len(test_sets[t]):>8d}{test_sets[t].query_symptoms.apply(len).mean():>12.1f}"
          f"{len(dev_sets[t]):>8d}{dev_sets[t].query_symptoms.apply(len).mean():>12.1f}")

print("\nSame test case across all four tiers:")
for t in TIERS:
    print(f"  [{t}] {test_sets[t].iloc[0]['query_text'][:130]}")
print(f"\n  TRUE: {test_sets['T0'].iloc[0]['true_disease']}")

## A4. Retrievers: SapBERT dense, BM25 sparse, IC-weighted ontology

In [ ]:
# =====================================================================
# A4a. Dense retrievers — SapBERT + MiniLM, with/without disease name
# =====================================================================
from sentence_transformers import SentenceTransformer
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print("encode device:", device)
if device == "cpu":
    print("[WARN] No GPU. Encoding will take ~10-15 min.")

DENSE_MODELS = {
    "sapbert": "cambridgeltl/SapBERT-from-PubMedBERT-fulltext",
    "minilm":  "all-MiniLM-L6-v2",
}
TEXT_VARIANTS = {"withname": "text_with_name", "noname": "text_no_name"}

encoders, EMB = {}, {}
for mkey, mname in DENSE_MODELS.items():
    encoders[mkey] = SentenceTransformer(mname, device=device)
    for vkey, col in TEXT_VARIANTS.items():
        cache = OUT + f"emb_{mkey}_{vkey}.npy"
        if os.path.exists(cache):
            EMB[(mkey, vkey)] = np.load(cache)
            print(f"  loaded cache {mkey}/{vkey} {EMB[(mkey, vkey)].shape}")
        else:
            print(f"  encoding {mkey}/{vkey} ...")
            e = encoders[mkey].encode(kb[col].tolist(), batch_size=64,
                                      show_progress_bar=True, convert_to_numpy=True,
                                      normalize_embeddings=True)
            np.save(cache, e); EMB[(mkey, vkey)] = e
            print(f"    saved {e.shape}")

def dense_scores(query_text, mkey="sapbert", vkey="noname"):
    q = encoders[mkey].encode([query_text], normalize_embeddings=True)
    return (EMB[(mkey, vkey)] @ q.T).ravel()

# free the encoders we do not need at query time to leave VRAM for MedGemma
del encoders["minilm"]
torch.cuda.empty_cache()
print("Dense retrievers ready (minilm encoder released; its embeddings are cached).")

In [ ]:
# =====================================================================
# A4b. BM25 (self-contained)
# =====================================================================
class BM25:
    def __init__(self, corpus_tokens, k1=1.5, b=0.75):
        self.k1, self.b = k1, b
        self.N = len(corpus_tokens)
        self.doclen = np.array([len(d) for d in corpus_tokens], dtype=np.float32)
        self.avgdl = float(self.doclen.mean())
        dfc = defaultdict(int)
        self.postings = defaultdict(list)
        for i, d in enumerate(corpus_tokens):
            c = defaultdict(int)
            for t in d:
                c[t] += 1
            for t, f in c.items():
                dfc[t] += 1
                self.postings[t].append((i, f))
        self.idf = {t: math.log(1 + (self.N - n + 0.5) / (n + 0.5)) for t, n in dfc.items()}

    def scores(self, query_tokens):
        s = np.zeros(self.N, dtype=np.float32)
        for t in query_tokens:
            if t not in self.postings:
                continue
            idf = self.idf[t]
            for i, f in self.postings[t]:
                s[i] += idf * (f * (self.k1 + 1)) / (
                    f + self.k1 * (1 - self.b + self.b * self.doclen[i] / self.avgdl))
        return s


STOP = {"of", "the", "a", "an", "and", "or", "in", "with", "abnormality",
        "abnormal", "increased", "decreased", "morphology"}

def tokenize(text):
    toks = re.findall(r"[a-z0-9]+", str(text).lower())
    return [t for t in toks if t not in STOP and len(t) > 2]

bm25 = BM25([tokenize(t) for t in kb["text_no_name"]])
print(f"BM25 index: {bm25.N:,} docs, {len(bm25.postings):,} unique tokens")

def bm25_scores(query_text):
    return bm25.scores(tokenize(query_text))

In [ ]:
# =====================================================================
# A4c. IC-weighted phenotype overlap (Phrank-style), sparse + vectorised
#      v3 looped in Python over every disease for every query. With 400 test
#      calls plus a weight grid over 240 dev cases that becomes the bottleneck,
#      so this builds a CSR matrix once and does a single sparse product.
# =====================================================================
from scipy import sparse

_term_ids, _rows, _cols, _vals = {}, [], [], []
for i, (ids, ws) in enumerate(zip(kb["hpo_ids"], kb["symptom_weights"])):
    prop = {}
    for h, w in zip(ids, ws):
        for a in ancestors_of(h):
            prop[a] = max(prop.get(a, 0.0), w)
    for t, w in prop.items():
        j = _term_ids.setdefault(t, len(_term_ids))
        _rows.append(i); _cols.append(j)
        _vals.append(w * IC.get(t, 0.0))        # fold IC into the stored weight

IC_MAT = sparse.csr_matrix((_vals, (_rows, _cols)),
                           shape=(len(kb), len(_term_ids)), dtype=np.float32)
print(f"IC matrix: {IC_MAT.shape[0]:,} x {IC_MAT.shape[1]:,}, "
      f"{IC_MAT.nnz:,} nnz ({IC_MAT.nnz/IC_MAT.shape[0]:.0f} terms/disease)")

# normalise by sqrt(disease breadth) so 186-symptom diseases do not dominate
_IC_NORM = np.sqrt(np.asarray(IC_MAT.sum(axis=1)).ravel())
_IC_NORM[_IC_NORM == 0] = 1.0


def ic_scores(query_hpo_ids):
    q = set()
    for h in query_hpo_ids:
        q |= ancestors_of(h)
    cols = [_term_ids[t] for t in q if t in _term_ids]
    if not cols:
        return np.zeros(len(kb), dtype=np.float32)
    v = np.zeros(IC_MAT.shape[1], dtype=np.float32)
    v[cols] = 1.0
    return (IC_MAT @ v) / _IC_NORM


# equivalence check against the v3 formulation on a few cases
def _ic_scores_reference(query_hpo_ids):
    q = set()
    for h in query_hpo_ids:
        q |= ancestors_of(h)
    out = np.zeros(len(kb), dtype=np.float64)
    for i, (ids, ws) in enumerate(zip(kb["hpo_ids"], kb["symptom_weights"])):
        prop = {}
        for h, w in zip(ids, ws):
            for a in ancestors_of(h):
                prop[a] = max(prop.get(a, 0.0), w)
        out[i] = sum(IC.get(t, 0.0) * prop[t] for t in q if t in prop)
    return out / _IC_NORM

_c = test_sets["T0"].iloc[0]
_fast = ic_scores(_c["query_hpo_ids"])
_ref  = _ic_scores_reference(_c["query_hpo_ids"])
_dev  = float(np.abs(_fast - _ref).max())
print(f"vectorised vs v3 reference: max abs diff = {_dev:.2e} "
      f"({'OK' if _dev < 1e-3 else 'MISMATCH — investigate before trusting results'})")

In [ ]:
# =====================================================================
# A4d. Reciprocal Rank Fusion + unified retrieve()
# =====================================================================
def _ranks_from_scores(s):
    s = np.asarray(s)
    n = len(s)
    order = np.argsort(-s)
    r = np.empty(n, dtype=np.int32)
    r[order] = np.arange(1, n + 1)
    return r


def rrf(score_arrays, k=60, weights=None):
    weights = weights or [1.0] * len(score_arrays)
    fused = np.zeros(len(score_arrays[0]), dtype=np.float64)
    for arr, w in zip(score_arrays, weights):
        if w:
            fused += w / (k + _ranks_from_scores(arr))
    return fused


# Frozen fusion config. A4e overwrites this from the DEV split; the default here
# is the v3 equal-weight setting so the cell is runnable before tuning.
BEST_RRF = {"weights": [1.0, 1.0, 1.0], "k": 60}


def signal_scores(case):
    """The three base signals, in the fixed order [dense, bm25, ic]."""
    return [dense_scores(case["query_text"], "sapbert", "noname"),
            bm25_scores(case["query_text"]),
            ic_scores(case["query_hpo_ids"])]


def retrieve(case, method="hybrid_tuned", top_k=TOP_K_RETRIEVE):
    """case: a row with query_text + query_hpo_ids. Returns (indices, scores)."""
    qt, qids = case["query_text"], case["query_hpo_ids"]

    if   method == "sapbert":      s = dense_scores(qt, "sapbert", "noname")
    elif method == "sapbert_name": s = dense_scores(qt, "sapbert", "withname")
    elif method == "minilm":
        raise ValueError(
            "minilm's encoder was released in A4a to free VRAM for MedGemma, so it is "
            "not a query-time option. Its cached embeddings are still in EMB. It was "
            "the weakest dense variant in v3 anyway (R@5 0.67/0.42/0.59/0.25).")
    elif method == "bm25":         s = bm25_scores(qt)
    elif method == "ic":           s = ic_scores(qids)
    elif method == "hybrid":
        s = rrf(signal_scores(case))
    elif method == "hybrid_dense_bm25":
        s = rrf([dense_scores(qt, "sapbert", "noname"), bm25_scores(qt)])
    elif method == "hybrid_dense_ic":
        s = rrf([dense_scores(qt, "sapbert", "noname"), ic_scores(qids)])
    elif method == "hybrid_tuned":
        s = rrf(signal_scores(case), k=BEST_RRF["k"], weights=BEST_RRF["weights"])
    else:
        raise ValueError(method)

    idx = np.argsort(-s)[:top_k]
    return idx, s[idx]


# minilm dropped from METHODS: its encoder was released in A4a to save VRAM and it
# was the weakest dense variant in v3 anyway (R@5 0.67/0.42/0.59/0.25).
METHODS = ["sapbert_name", "sapbert", "bm25", "ic",
           "hybrid_dense_bm25", "hybrid_dense_ic", "hybrid", "hybrid_tuned"]

c = test_sets["T0"].iloc[0]
for m in ("sapbert", "bm25", "ic", "hybrid"):
    idx, _ = retrieve(c, m, top_k=5)
    names = kb.iloc[idx]["disease_name"].tolist()
    print(f"{m:20s} {'HIT' if c['true_disease'] in names else '   '}  {names[0][:55]}")
print(f"{'':20s}TRUE: {c['true_disease']}")

## A4e. Freeze the fusion weights on DEV

Unchanged from v4. Note that this cell is now *evidence for* the paper's second claim rather than a tuning step whose result we trust: the weights it selects are the ones that transfer worst externally.

In [ ]:
# =====================================================================
# A4e. Grid-search RRF weights + k on DEV. Frozen afterwards.
# =====================================================================
K_TUNE   = TOP_K_RETRIEVE      # primary objective: recall at the candidate-set size
K_TIE    = 5                   # tie-break

print("Precomputing base signals on the dev split ...")
t0 = time.time()
DEV_RANKS, DEV_TRUEROW = {}, {}
name_to_rows = defaultdict(list)
for i, nm in enumerate(kb["disease_name"]):
    name_to_rows[nm].append(i)

for tier in TIERS:
    d = dev_sets[tier]
    rr, tt = [], []
    for _, row in d.iterrows():
        rr.append([_ranks_from_scores(s) for s in signal_scores(row)])
        tt.append(set(name_to_rows[row["true_disease"]]))
    DEV_RANKS[tier], DEV_TRUEROW[tier] = rr, tt
print(f"  done in {time.time()-t0:.0f}s "
      f"({sum(len(v) for v in DEV_RANKS.values())} cases x 3 signals)")


def _fuse(ranks, weights, k):
    f = np.zeros(len(ranks[0]), dtype=np.float64)
    for r, w in zip(ranks, weights):
        if w:
            f += w / (k + r)
    return f


def _dev_recall(weights, k, ks=(K_TIE, K_TUNE)):
    """Mean-over-tiers recall@k for a candidate weight vector."""
    per_k = {kk: [] for kk in ks}
    for tier in TIERS:
        hits = {kk: [] for kk in ks}
        for ranks, true_rows in zip(DEV_RANKS[tier], DEV_TRUEROW[tier]):
            fused = _fuse(ranks, weights, k)
            top = np.argsort(-fused)[:max(ks)]
            for kk in ks:
                hits[kk].append(bool(true_rows & set(top[:kk].tolist())))
        for kk in ks:
            per_k[kk].append(float(np.mean(hits[kk])))
    return {kk: float(np.mean(v)) for kk, v in per_k.items()}, per_k


GRID_W_DENSE = [0.0, 0.25, 0.5, 1.0]
GRID_W_BM25  = [1.0, 2.0, 3.0, 4.0]
GRID_W_IC    = [0.0, 0.25, 0.5, 1.0, 2.0]
GRID_K       = [10, 30, 60]

print(f"\nSearching {len(GRID_W_DENSE)*len(GRID_W_BM25)*len(GRID_W_IC)*len(GRID_K)} "
      f"configurations on DEV ...")
t0 = time.time()
tuning_rows = []
for wd in GRID_W_DENSE:
    for wb in GRID_W_BM25:
        for wi in GRID_W_IC:
            if wd == 0 and wi == 0:
                continue                       # degenerates to plain BM25
            for k in GRID_K:
                m, _ = _dev_recall([wd, wb, wi], k)
                tuning_rows.append({"w_dense": wd, "w_bm25": wb, "w_ic": wi, "k": k,
                                    f"R@{K_TUNE}": m[K_TUNE], f"R@{K_TIE}": m[K_TIE]})
tune = pd.DataFrame(tuning_rows).sort_values(
    [f"R@{K_TUNE}", f"R@{K_TIE}"], ascending=False).reset_index(drop=True)
print(f"  done in {time.time()-t0:.0f}s")

best = tune.iloc[0]
BEST_RRF = {"weights": [float(best.w_dense), float(best.w_bm25), float(best.w_ic)],
            "k": int(best.k)}

print(f"\nTop 8 configurations on DEV (dense / bm25 / ic weights):")
print(tune.head(8).to_string(index=False))
print(f"\nFROZEN: BEST_RRF = {BEST_RRF}")

# --- how does the tuned fusion compare with the fixed single-signal baselines? ---
print(f"\nDEV mean recall, single weight vector across all four tiers:")
print(f"{'method':22s}{'R@'+str(K_TUNE):>10s}{'R@'+str(K_TIE):>10s}")
_cands = {
    "bm25 only":            ([0.0, 1.0, 0.0], 60),
    "ic only":              ([0.0, 0.0, 1.0], 60),
    "dense only":           ([1.0, 0.0, 0.0], 60),
    "hybrid (v3, equal)":   ([1.0, 1.0, 1.0], 60),
    "hybrid_tuned":         (BEST_RRF["weights"], BEST_RRF["k"]),
}
_dev_summary = {}
for nm, (w, k) in _cands.items():
    m, per = _dev_recall(w, k)
    _dev_summary[nm] = (m, per)
    print(f"{nm:22s}{m[K_TUNE]:>10.3f}{m[K_TIE]:>10.3f}")

print(f"\nPer-tier R@{K_TUNE} on DEV:")
print(f"{'method':22s}" + "".join(f"{t:>8s}" for t in TIERS))
for nm, (m, per) in _dev_summary.items():
    print(f"{nm:22s}" + "".join(f"{v:>8.3f}" for v in per[K_TUNE]))

# --- choose the retriever for Part B, on DEV evidence only ---------------
_ranked = sorted(_dev_summary.items(), key=lambda kv: (-kv[1][0][K_TUNE], -kv[1][0][K_TIE]))
_winner = _ranked[0][0]
RETRIEVER = "hybrid_tuned" if _winner == "hybrid_tuned" else {
    "bm25 only": "bm25", "ic only": "ic", "dense only": "sapbert",
    "hybrid (v3, equal)": "hybrid"}[_winner]

print(f"\n>>> RETRIEVER = '{RETRIEVER}'  (selected on DEV, by mean R@{K_TUNE})")
if RETRIEVER == "bm25":
    print(">>> NOTE: plain BM25 won on dev. Given the lexical-leakage concern in the")
    print(">>> header, report this explicitly and discuss it — do not present it as a")
    print(">>> clean retrieval result.")

tune.to_csv(OUT + "table_rrf_tuning_dev.csv", index=False)
with open(OUT + "best_rrf.json", "w") as f:
    json.dump({"BEST_RRF": BEST_RRF, "RETRIEVER": RETRIEVER,
               "objective": f"mean R@{K_TUNE} over tiers, tie-break R@{K_TIE}",
               "n_dev_diseases": int(len(dev_idx))}, f, indent=2)
print("\nSaved table_rrf_tuning_dev.csv and best_rrf.json")

## D1. Phenotype-driven baselines

Phrank-style, LIRICAL-style likelihood ratio, and Resnik/Phenomizer one-sided semantic similarity. **Re-implementations** — read the cell header before citing them.

In [ ]:
# =====================================================================
# D1. Phenotype-driven baselines: Phrank-style, LIRICAL-style LR, Resnik.
#
#     READ THIS BEFORE YOU CITE ANYTHING FROM THIS CELL.
#
#     These are RE-IMPLEMENTATIONS written from the published descriptions of
#     the methods. They are NOT the authors' code and they have NOT been
#     validated against the original implementations. In the paper you must:
#       (a) call them "Phrank-style", "LIRICAL-style", "Resnik/Phenomizer-style"
#           or "our re-implementation of X", never plain "Phrank"/"LIRICAL";
#       (b) print the formulas below in the supplement;
#       (c) state that absolute values are not comparable to the numbers in the
#           original papers, which use different knowledge bases and, for
#           LIRICAL/Exomiser, genotype evidence this pipeline does not have.
#     If a reviewer asks for the real tools, LIRICAL and Exomiser are Java and
#     need multi-GB data bundles — run them offline, not in this notebook.
#
#     FORMULAS AS IMPLEMENTED
#     -----------------------
#     Let closure(X) be the union of HPO ancestor sets of the terms in X, IC(t) =
#     -log(n(t)/N) with n(t) = number of KB diseases whose closure contains t.
#
#     Phrank-style:
#         S(Q,D) = sum over t in closure(Q) ∩ closure(D) of IC(t)
#       Unweighted and unnormalised — deliberately different from this pipeline's
#       own `ic` scorer, which additionally applies Orphanet frequency weights and
#       divides by sqrt(disease breadth).
#
#     Resnik / Phenomizer-style (one-sided, query -> disease):
#         S(Q,D) = mean over q in Q of  max over t in anc(q) ∩ closure(D) of IC(t)
#       The inner max is exactly IC of the most informative common ancestor
#       between q and the best-matching annotation of D.
#
#     LIRICAL-style phenotype-only likelihood ratio:
#         For a query term q and disease D, let LR(t) = P(t|D) / P(t|background),
#         with P(t|D) the Orphanet frequency weight of t in D (propagated to
#         ancestors by max) and P(t|background) = n(t)/N.
#             exact match      : LR_ex  = LR(q)                       if q in D
#             ancestor match   : LR_anc = ANC_DISCOUNT * max over a in
#                                anc(q)\{q} of LR(a)                  if any a in D
#             no match         : LR_0   = FN_RATE / P(q|background)
#         S(Q,D) = sum over q of log10( max(LR_ex, LR_anc, LR_0) )
#       Each matched term is scored against ITS OWN background frequency. That is
#       what stops a generic ancestor from beating an exact match: a common
#       ancestor has a large P(t|background), so its LR is near 1 and contributes
#       almost nothing. An earlier version of this cell compared every match
#       against the *query term's* background and consequently ranked diseases
#       that shared only a generic ancestor above the true disease.
#       This is still a simplification: real LIRICAL derives the ancestor fallback
#       from the ontology structure rather than a flat discount, and folds in
#       genotype likelihood ratios. Say so.
# =====================================================================
ANC_DISCOUNT = 0.5
EPS_LR       = 1.0 / (2 * N_DIS)

# Weighted and binary closure matrices over the same term index as IC_MAT.
_r, _c2, _vw = [], [], []
for i, (ids, ws) in enumerate(zip(kb["hpo_ids"], kb["symptom_weights"])):
    prop = {}
    for h, w in zip(ids, ws):
        for a in ancestors_of(h):
            prop[a] = max(prop.get(a, 0.0), float(w))
    for t, w in prop.items():
        j = _term_ids.get(t)
        if j is None:
            continue
        _r.append(i); _c2.append(j); _vw.append(w)

CLO_W   = sparse.csr_matrix((_vw, (_r, _c2)),
                            shape=(len(kb), len(_term_ids)), dtype=np.float32)
CLO_BIN = CLO_W.copy()
CLO_BIN.data = np.ones_like(CLO_BIN.data)
CLO_W_CSC, CLO_BIN_CSC = CLO_W.tocsc(), CLO_BIN.tocsc()

IC_VEC = np.zeros(len(_term_ids), dtype=np.float32)
for t, j in _term_ids.items():
    IC_VEC[j] = IC.get(t, 0.0)

BG_VEC = np.full(len(_term_ids), EPS_LR, dtype=np.float32)
for t, j in _term_ids.items():
    BG_VEC[j] = max(term_doc_count.get(t, 1) / N_DIS, EPS_LR)

print(f"closure matrices: {CLO_W.shape[0]:,} x {CLO_W.shape[1]:,}, "
      f"{CLO_W.nnz:,} nnz")


def _q_closure_cols(qids):
    q = set()
    for h in qids:
        q |= ancestors_of(h)
    return [_term_ids[t] for t in q if t in _term_ids]


def phrank_scores(qids):
    cols = _q_closure_cols(qids)
    if not cols:
        return np.zeros(len(kb), dtype=np.float32)
    v = np.zeros(len(_term_ids), dtype=np.float32)
    v[cols] = IC_VEC[cols]
    return np.asarray(CLO_BIN @ v).ravel()


def resnik_scores(qids):
    acc, n_used = np.zeros(len(kb), dtype=np.float32), 0
    for h in qids:
        cols = [_term_ids[t] for t in ancestors_of(h) if t in _term_ids]
        if not cols:
            continue
        sub = CLO_BIN_CSC[:, cols].multiply(IC_VEC[cols][None, :])
        acc += np.asarray(sub.max(axis=1).todense()).ravel()
        n_used += 1
    return acc / max(1, n_used)


# per-term likelihood ratios: LR[i, j] = P(term j | disease i) / P(term j | bg)
LR_MAT_CSC = CLO_W.multiply(sparse.csr_matrix(1.0 / BG_VEC)).tocsc()


def lirical_scores(qids):
    score = np.zeros(len(kb), dtype=np.float64)
    for h in qids:
        hc = canon(h)
        j_ex = _term_ids.get(hc)
        cols_anc = [_term_ids[t] for t in ancestors_of(h)
                    if t in _term_ids and _term_ids[t] != j_ex]

        lr_ex = (np.asarray(LR_MAT_CSC[:, [j_ex]].todense()).ravel()
                 if j_ex is not None else np.zeros(len(kb)))
        lr_anc = (ANC_DISCOUNT *
                  np.asarray(LR_MAT_CSC[:, cols_anc].max(axis=1).todense()).ravel()
                  if cols_anc else np.zeros(len(kb)))

        bg_q = float(BG_VEC[j_ex]) if j_ex is not None else 1.0
        lr_0 = EPS_LR / max(bg_q, EPS_LR)          # false-negative penalty

        lr = np.maximum(np.maximum(lr_ex, lr_anc), lr_0)
        score += np.log10(np.maximum(lr, EPS_LR))
    return score.astype(np.float32)


BASELINE_SCORERS = {
    "phrank_style":  lambda c: phrank_scores(c["query_hpo_ids"]),
    "resnik_style":  lambda c: resnik_scores(c["query_hpo_ids"]),
    "lirical_style": lambda c: lirical_scores(c["query_hpo_ids"]),
}

# Wrap retrieve() so the baselines are first-class methods everywhere.
if "_retrieve_core" not in globals():
    _retrieve_core = retrieve


def retrieve(case, method="hybrid_tuned", top_k=TOP_K_RETRIEVE):
    if method in BASELINE_SCORERS:
        s = np.asarray(BASELINE_SCORERS[method](case))
        idx = np.argsort(-s)[:top_k]
        return idx, s[idx]
    return _retrieve_core(case, method, top_k)


BASELINE_METHODS = list(BASELINE_SCORERS)
METHODS_V5 = METHODS + BASELINE_METHODS

# ---- sanity: a synthetic query from a disease's own annotations must rank it top
_ok = 0
_probe_idx = [int(x) for x in np.linspace(0, len(kb) - 1, 4).astype(int)]
for _g in _probe_idx:
    _kr = kb.iloc[_g]
    _n = min(6, len(_kr["hpo_ids"]))
    _syn = {"query_hpo_ids": list(_kr["hpo_ids"])[:_n],
            "query_symptoms": list(_kr["symptoms"])[:_n],
            "query_text": "Patient presents with: " + ", ".join(list(_kr["symptoms"])[:_n])}
    line = []
    for m in BASELINE_METHODS:
        idx, _ = retrieve(_syn, m, top_k=25)
        rank = next((r + 1 for r, v in enumerate(idx) if int(v) == _g), ">25")
        line.append(f"{m}={rank}")
        _ok += (rank == 1)
    print(f"  {_kr['disease_name'][:38]:40s} " + "  ".join(line))
print(f"\nself-retrieval sanity: {_ok}/{4*len(BASELINE_METHODS)} at rank 1")
print("(low numbers here mean a baseline is broken — fix before scoring anything)")


## A5. Retrieval ablation on the internal TEST set (no API calls)

In [ ]:
# =====================================================================
# A5. Retrieval ablation: all methods x 4 tiers  (NO API CALLS)
# =====================================================================
K_VALUES = (1, 3, 5, 10, TOP_K_RETRIEVE)

def eval_retrieval(test_df, method, k_values=K_VALUES, top_k=TOP_K_RETRIEVE):
    hits = {k: [] for k in k_values}
    ranks = []
    for _, row in test_df.iterrows():
        idx, _ = retrieve(row, method, top_k=top_k)
        names = kb.iloc[idx]["disease_name"].tolist()
        try:
            r = names.index(row["true_disease"]) + 1
        except ValueError:
            r = None
        ranks.append(r)
        for k in k_values:
            hits[k].append(bool(r is not None and r <= k))
    return {k: float(np.mean(v)) for k, v in hits.items()}, hits, ranks


retrieval_results, retrieval_hits = {}, {}
for tier in TIERS:
    for m in METHODS_V5:
        if m == "minilm":
            continue
        rec, hits, ranks = eval_retrieval(test_sets[tier], m)
        retrieval_results[(tier, m)] = rec
        retrieval_hits[(tier, m)] = hits
        print(f"{tier} {m:20s} " + "  ".join(f"R@{k}={rec[k]:.3f}" for k in K_VALUES))
    print("-" * 82)

with open(OUT + "retrieval_ablation.pkl", "wb") as f:
    pickle.dump({"results": retrieval_results, "hits": retrieval_hits,
                 "k_values": K_VALUES, "methods": METHODS_V5, "tiers": TIERS,
                 "best_rrf": BEST_RRF, "retriever": RETRIEVER}, f)

_avail = [m for m in METHODS_V5 if (TIERS[0], m) in retrieval_results]
tbl = pd.DataFrame(
    [{"tier": t, "method": m, **{f"R@{k}": retrieval_results[(t, m)][k] for k in K_VALUES}}
     for t in TIERS for m in _avail])
tbl.to_csv(OUT + "table_retrieval_ablation.csv", index=False)

print(f"\nRecall@{TOP_K_RETRIEVE} by tier — this is the CEILING for the LLM stage:")
print(tbl.pivot(index="method", columns="tier", values=f"R@{TOP_K_RETRIEVE}").round(3).to_string())
print("\nRecall@5 by tier:")
print(tbl.pivot(index="method", columns="tier", values="R@5").round(3).to_string())
print(f"\nRETRIEVER in use for Part B: {RETRIEVER}")

In [ ]:
# =====================================================================
# A5b. Paired significance on the TEST set
# =====================================================================
from scipy.stats import binomtest

def mcnemar(hits_a, hits_b):
    """Exact McNemar. Returns (p, n_a_only, n_b_only)."""
    b = sum(1 for x, y in zip(hits_a, hits_b) if x and not y)
    c = sum(1 for x, y in zip(hits_a, hits_b) if y and not x)
    if b + c == 0:
        return 1.0, b, c
    return binomtest(b, b + c, 0.5).pvalue, b, c


def wilson(hits, z=1.96):
    n = len(hits)
    if n == 0:
        return (0.0, 0.0)
    p = sum(hits) / n
    d = 1 + z**2 / n
    c = (p + z**2 / (2*n)) / d
    h = z * math.sqrt(p*(1-p)/n + z**2/(4*n**2)) / d
    return (max(0, c-h), min(1, c+h))


print(f"Tuned fusion vs the v3 equal-weight hybrid, at R@{TOP_K_RETRIEVE} (paired):")
print(f"{'tier':6s}{'tuned':>9s}{'v3 equal':>10s}{'delta':>9s}{'p':>10s}{'95% CI (tuned)':>22s}")
for tier in TIERS:
    a = retrieval_hits[(tier, "hybrid_tuned")][TOP_K_RETRIEVE]
    b = retrieval_hits[(tier, "hybrid")][TOP_K_RETRIEVE]
    p, nb, nc = mcnemar(a, b)
    reg_p("F1_internal_retrieval", f"{tier}: hybrid_tuned vs hybrid (R@25)", p)
    lo, hi = wilson(a)
    print(f"{tier:6s}{np.mean(a):>9.3f}{np.mean(b):>10.3f}{np.mean(a)-np.mean(b):>+9.3f}"
          f"{p:>10.4f}{f'[{lo:.3f}, {hi:.3f}]':>22s}")

print(f"\nTuned fusion vs plain BM25, at R@{TOP_K_RETRIEVE} (paired):")
print(f"{'tier':6s}{'tuned':>9s}{'bm25':>10s}{'delta':>9s}{'p':>10s}")
for tier in TIERS:
    a = retrieval_hits[(tier, "hybrid_tuned")][TOP_K_RETRIEVE]
    b = retrieval_hits[(tier, "bm25")][TOP_K_RETRIEVE]
    p, nb, nc = mcnemar(a, b)
    reg_p("F1_internal_retrieval", f"{tier}: hybrid_tuned vs bm25 (R@25)", p)
    print(f"{tier:6s}{np.mean(a):>9.3f}{np.mean(b):>10.3f}{np.mean(a)-np.mean(b):>+9.3f}{p:>10.4f}")

print("\nDisease-name ablation (does the name in the indexed text help?):")
for tier in TIERS:
    wn = retrieval_hits[(tier, "sapbert_name")][5]
    nn = retrieval_hits[(tier, "sapbert")][5]
    p, b, c = mcnemar(wn, nn)
    print(f"  {tier}: with_name={np.mean(wn):.3f}  no_name={np.mean(nn):.3f}  p={p:.4f}")

In [ ]:
# =====================================================================
# A5c. Degradation curves
# =====================================================================
import matplotlib.pyplot as plt

SHOW = ["sapbert", "bm25", "ic", "hybrid", "hybrid_tuned"]
LABEL = {"sapbert": "SapBERT (dense)", "bm25": "BM25 (sparse)",
         "ic": "IC-weighted ontology", "hybrid": "Hybrid RRF (equal)",
         "hybrid_tuned": "Hybrid RRF (dev-tuned)"}

fig, axes = plt.subplots(1, 3, figsize=(16, 4.3))
for ax, k in zip(axes, (3, 5, TOP_K_RETRIEVE)):
    for m in SHOW:
        if (TIERS[0], m) not in retrieval_results:
            continue
        ys = [retrieval_results[(t, m)][k] for t in TIERS]
        ax.plot(TIERS, ys, marker="o", label=LABEL[m])
    ax.set_title(f"Recall@{k} under phenotypic noise"
                 + ("  (LLM ceiling)" if k == TOP_K_RETRIEVE else ""))
    ax.set_xlabel("noise tier"); ax.set_ylabel(f"Recall@{k}")
    ax.set_ylim(0, 1.02); ax.grid(alpha=.3)
axes[0].legend(fontsize=8)
plt.tight_layout()
plt.savefig(OUT + "fig_degradation_retrieval.png", dpi=200, bbox_inches="tight")
plt.show()
print("Saved fig_degradation_retrieval.png")

# Part B — the LLM stage (internal)

Gemini stays in the **internal** ensemble: v4's checkpoint holds 400/400 for all three members, so there is no reason to discard it here. It is dropped only from the external arm, where 834 calls would exceed the 500 RPD free-tier cap.

In [ ]:
# =====================================================================
# B0. Quota model + budget calculator
#     VERIFY AGAINST YOUR OWN DASHBOARDS. Free-tier limits change.
# =====================================================================
LIMITS = {
    # --- slot A: local. No provider, no meter, no pacing. ---
    "google/medgemma-27b-text-it":
        dict(provider="local",   rpm=None, rpd=None,  tpm=None,   tpd=None),

    # --- slot B: Google AI Studio free tier ---
    #   v3 measured 563 tok/call and completed 400/400 in a single day.
    "gemini-3.5-flash-lite":
        dict(provider="gemini",  rpm=15,   rpd=500,   tpm=250000, tpd=None),

    # --- slot C: Mistral la Plateforme free (Experiment) tier ---
    #   Mistral documents ~1 request/second, 500k tokens/minute and 1B
    #   tokens/MONTH. There is no published tokens-per-DAY cap, so tpd=None.
    #   v3 had tpd=200000 here, copied from the Groq console. That false cap is
    #   exactly what halted T3 at 31/100 with 190k "used" of a limit that does
    #   not exist. rpm/tpm are set below the documented ceiling for safety.
    "mistral-small-latest":
        dict(provider="mistral", rpm=50,   rpd=50000, tpm=480000, tpd=None),
}

ENSEMBLE = ENSEMBLE_INTERNAL     # set in K0c

EST_TOKENS_PER_CALL  = 600     # v3 measured: gemini 563, mistral 575 at TOP_K=25
MAX_COMPLETION_TOKENS = 1024   # API models
# LOCAL_MAX_NEW_TOKENS is set in K0c (200)


def budget(model, est=EST_TOKENS_PER_CALL):
    """(calls available per day, which limit binds, calls allowed per minute)."""
    L = LIMITS[model]
    if L["provider"] == "local":
        return 10**9, "none", 10**6
    caps = {"RPD": L["rpd"]} if L.get("rpd") else {}
    if L.get("tpd"):
        caps["TPD"] = L["tpd"] // est
    if not caps:
        caps = {"none": 10**9}
    binding = min(caps, key=caps.get)
    per_min = L["rpm"] or 10**6
    if L.get("tpm"):
        per_min = min(per_min, max(1, L["tpm"] // est))
    return caps[binding], binding, max(1, per_min)


n_evals = N_TEST * len(TIERS)
print(f"Workload: {N_TEST} cases x {len(TIERS)} tiers = {n_evals} evaluations per model\n")
print(f"{'model':32s}{'provider':10s}{'calls/day':>11s}{'binds':>7s}{'calls/min':>11s}{'days':>7s}")
for m in ENSEMBLE:
    n, why, pm = budget(m)
    days = n_evals / n
    print(f"{m:32s}{LIMITS[m]['provider']:10s}{min(n,999999):>11d}{why:>7s}{min(pm,999999):>11d}"
          f"{days:>7.2f}")

_pace_secs = sum(n_evals * 60.0 / budget(m)[2] for m in ENSEMBLE if LIMITS[m]["provider"] != "local")
print(f"\nAPI pacing alone: ~{_pace_secs/60:.0f} min across the whole run.")
print("MedGemma generation dominates: budget ~2-4 h for all 400 cases.")
print("\nAll three slots sit on independent budgets, so none competes with another.")

In [ ]:
# =====================================================================
# B1. Provider clients + call_llm (api + local dispatch)
# =====================================================================
from google import genai
from google.genai import types

# mistralai moved this class between versions; accept either path.
try:
    from mistralai import Mistral
except ImportError:
    from mistralai.client import Mistral

def _get_secret(name, prompt):
    """Kaggle Secrets first, then env, then interactive."""
    try:
        from kaggle_secrets import UserSecretsClient
        v = UserSecretsClient().get_secret(name)
        if v:
            print(f"  {name}: from Kaggle Secrets")
            return v
    except Exception:
        pass
    if os.environ.get(name):
        print(f"  {name}: from environment")
        return os.environ[name]
    from getpass import getpass
    return getpass(prompt)

print("Loading credentials ...")
GEMINI_API_KEY  = _get_secret("GEMINI_API_KEY",  "Gemini API key: ")
MISTRAL_API_KEY = _get_secret("MISTRAL_API_KEY", "Mistral API key: ")

gemini_client  = genai.Client(api_key=GEMINI_API_KEY)
mistral_client = Mistral(api_key=MISTRAL_API_KEY)

# populated by B2b; call_llm refuses local models until then
LOCAL = {}

USAGE = defaultdict(lambda: {"calls": 0, "tokens": 0})
_last_call = defaultdict(float)


class QuotaExhausted(Exception):
    pass


def _pace(model):
    if LIMITS[model]["provider"] == "local":
        return
    _, _, per_min = budget(model)
    interval = 60.0 / per_min
    dt = time.time() - _last_call[model]
    if dt < interval:
        time.sleep(interval - dt)
    _last_call[model] = time.time()


def _guard(model):
    """Refuse a call we know will 429. Cheaper than burning retries."""
    L, u = LIMITS[model], USAGE[model]
    if L["provider"] == "local":
        return
    if L.get("rpd") and u["calls"] >= L["rpd"] * 0.97:
        raise QuotaExhausted(f"{model}: RPD {u['calls']}/{L['rpd']}")
    if L.get("tpd") and u["tokens"] >= L["tpd"] * 0.95:
        raise QuotaExhausted(f"{model}: TPD {u['tokens']:,}/{L['tpd']:,}")


def _is_rate_limit(e):
    s = str(e).lower()
    return any(x in s for x in ("429", "rate limit", "rate_limit", "resource exhausted",
                                "too many requests", "quota"))


RANK_SCHEMA = {
    "type": "OBJECT",
    "properties": {"ranking": {"type": "ARRAY", "items": {"type": "INTEGER"}}},
    "required": ["ranking"],
}


def _call_local(model, system_prompt, user_prompt):
    """Deterministic greedy generation from a locally loaded HF model."""
    if model not in LOCAL:
        raise RuntimeError(f"{model} is not loaded. Run cell B2b first.")
    tok, mdl = LOCAL[model]["tok"], LOCAL[model]["model"]
    msgs = [{"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_prompt}]
    try:
        text = tok.apply_chat_template(msgs, add_generation_prompt=True, tokenize=False)
    except Exception:
        # some Gemma templates reject a separate system turn — merge it
        text = tok.apply_chat_template(
            [{"role": "user", "content": system_prompt + "\n\n" + user_prompt}],
            add_generation_prompt=True, tokenize=False)
    enc = tok(text, return_tensors="pt", add_special_tokens=False).to(mdl.device)
    n_in = enc["input_ids"].shape[-1]
    with torch.no_grad():
        out = mdl.generate(
            **enc,
            max_new_tokens=LOCAL_MAX_NEW_TOKENS,
            do_sample=False, num_beams=1,          # deterministic
            pad_token_id=(tok.pad_token_id if tok.pad_token_id is not None
                          else tok.eos_token_id),
        )
    new = out[0][n_in:]
    return tok.decode(new, skip_special_tokens=True), n_in + int(new.shape[-1])


def call_llm(model, system_prompt, user_prompt, max_retries=3):
    _guard(model)
    _pace(model)
    prov = LIMITS[model]["provider"]
    last, attempt = None, 0
    while attempt < max_retries:
        try:
            if prov == "local":
                txt, tok_used = _call_local(model, system_prompt, user_prompt)

            elif prov == "mistral":
                r = mistral_client.chat.complete(
                    model=model, temperature=0.0,
                    max_tokens=MAX_COMPLETION_TOKENS,          # NOT max_completion_tokens
                    response_format={"type": "json_object"},
                    messages=[{"role": "system", "content": system_prompt},
                              {"role": "user",   "content": user_prompt}],
                )
                ch = r.choices[0]
                txt = ch.message.content
                used = getattr(r, "usage", None)
                tok_used = used.total_tokens if used else EST_TOKENS_PER_CALL

            elif prov == "gemini":
                cfg = types.GenerateContentConfig(
                    temperature=0.0, max_output_tokens=MAX_COMPLETION_TOKENS,
                    system_instruction=system_prompt,
                    response_mime_type="application/json",
                    response_schema=RANK_SCHEMA)
                r = gemini_client.models.generate_content(
                    model=model, contents=user_prompt, config=cfg)
                txt = r.text
                um = getattr(r, "usage_metadata", None)
                tok_used = getattr(um, "total_token_count", None) or EST_TOKENS_PER_CALL

            else:
                raise ValueError(f"unknown provider {prov!r}")

            if not txt or not txt.strip():
                raise RuntimeError("empty response")

            txt = re.sub(r"(?s)^.*?</think>", "", txt)   # strip leaked reasoning if any
            USAGE[model]["calls"]  += 1
            USAGE[model]["tokens"] += int(tok_used)
            return txt

        except Exception as e:
            last = e
            if prov != "local" and _is_rate_limit(e):
                USAGE[model]["tokens"] = LIMITS[model]["tpd"] or 10**9
                USAGE[model]["calls"]  = max(USAGE[model]["calls"],
                                             (LIMITS[model]["rpd"] or 10**9))
                raise QuotaExhausted(f"{model}: provider 429 — {e}")
            attempt += 1
            time.sleep(2 * attempt)
    raise RuntimeError(f"{model} failed after {max_retries}: {last}")


def usage_report():
    rows = []
    for m, u in USAGE.items():
        L = LIMITS[m]
        rows.append({"model": m.split("/")[-1], "provider": L["provider"],
                     "calls": u["calls"], "rpd": L.get("rpd"),
                     "tokens": u["tokens"], "tpd": L.get("tpd"),
                     "tok/call": round(u["tokens"] / max(1, u["calls"]))})
    return pd.DataFrame(rows)


print("\nAPI clients ready (gemini / mistral). Nothing spent yet.")

In [ ]:
RETRIEVER = "hybrid_tuned"   # only used for B2's self-test smoke print, not the app

## B2. Prompt construction + strict index parsing

Unchanged from v3: the re-rankers emit candidate **numbers**, never disease names, so a
hallucinated name is structurally impossible and output is ~4x cheaper.

In [ ]:
# =====================================================================
# B2. Prompt + strict index parsing
# =====================================================================
RERANK_SYSTEM = """You are a clinical decision-support assistant for rare disease differential diagnosis.
You are given a patient's phenotypic features and a numbered list of candidate diseases retrieved
from Orphanet. Re-order ALL candidates from most to least likely given the features.
Some features may be missing, imprecise, or unrelated to the true diagnosis; weigh specific,
discriminative features more heavily than common non-specific ones.
Respond with JSON only, no prose: {"ranking": [<candidate numbers, most likely first>]}
Include every candidate number exactly once."""


def build_prompt(case, cand_idx):
    feats = "\n".join(f"- {s}" for s in case["query_symptoms"])
    cands = "\n".join(f"{i+1}. {kb.iloc[j]['disease_name']}" for i, j in enumerate(cand_idx))
    return f"""Patient phenotypic features:
{feats}

Candidate diseases:
{cands}

Return JSON only: {{"ranking": [<all {len(cand_idx)} numbers, most likely first>]}}"""


def _collect_ints(x, out):
    if isinstance(x, bool):
        return
    if isinstance(x, int):
        out.append(x)
    elif isinstance(x, float) and float(x).is_integer():
        out.append(int(x))
    elif isinstance(x, str):
        m = re.fullmatch(r"\s*(\d+)\s*", x)
        if m:
            out.append(int(m.group(1)))
    elif isinstance(x, dict):
        for v in x.values():
            _collect_ints(v, out)
    elif isinstance(x, (list, tuple)):
        for v in x:
            _collect_ints(v, out)


def parse_ranking(raw, n):
    """Parse {"ranking":[ints]} -> 0-based positions. Returns (order, n_missing).

    Raises if NOTHING usable came back, so the runner records an error instead of
    silently substituting the retrieval order and scoring it as an LLM result.
    """
    txt = str(raw).strip()
    txt = re.sub(r"^```(?:json)?\s*", "", txt, flags=re.I)
    txt = re.sub(r"\s*```$", "", txt)
    a, b = txt.find("{"), txt.rfind("}")
    if a != -1 and b != -1:
        txt = txt[a:b+1]

    nums = []
    try:
        obj = json.loads(txt)
        rank = obj.get("ranking") if isinstance(obj, dict) else obj
        if rank is None:
            rank = obj
        _collect_ints(rank, nums)
    except Exception:
        pass
    if not nums:
        nums = [int(x) for x in re.findall(r"\d+", txt)]
    if not nums:
        raise ValueError(f"no usable indices in response: {txt[:200]!r}")

    seen, order = set(), []
    for v in nums:
        p = v - 1
        if 0 <= p < n and p not in seen:
            order.append(p); seen.add(p)
    if not order:
        raise ValueError(f"no in-range indices (1..{n}) in response: {txt[:200]!r}")

    missing = [p for p in range(n) if p not in seen]
    order.extend(missing)                      # deterministic completion
    return order, len(missing)


LAST_RAW = {}

def rerank_one(model, case, cand_idx):
    raw = call_llm(model, RERANK_SYSTEM, build_prompt(case, cand_idx))
    LAST_RAW[model] = raw
    order, n_missing = parse_ranking(raw, len(cand_idx))
    return [int(cand_idx[p]) for p in order], n_missing


print("Prompt + parser ready.")
_c = test_sets["T0"].iloc[0]
_ci, _ = retrieve(_c, RETRIEVER, top_k=TOP_K_RETRIEVE)
_p = build_prompt(_c, [int(x) for x in _ci])
print(f"Example prompt is {len(_p)} chars / ~{len(_p)//4} tokens.")

## B2b. Load MedGemma 27B on 2x T4

v4 used `bnb_4bit_compute_dtype=torch.float32`, which is numerically safe but forgoes Turing's fp16 tensor cores. v5 tries fp16, runs the same finite-logit probe, and falls back to fp32 automatically if the probe fails. Whichever is used is recorded for the Methods section.

In [ ]:
import gc
import torch

if "LOCAL" in globals() and MEDGEMMA_ID in LOCAL:
    del LOCAL[MEDGEMMA_ID]

for name in ("med_model", "med_tok", "model", "tok"):
    if name in globals():
        del globals()[name]

gc.collect()
torch.cuda.empty_cache()

for i in range(torch.cuda.device_count()):
    print(f"GPU {i}: {torch.cuda.memory_allocated(i) / 2**30:.2f} GiB allocated")

In [ ]:
# =====================================================================
# B2b. Load MedGemma 27B, 4-bit NF4, fp16 compute, eager attention
# =====================================================================
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

SRC = MEDGEMMA_PATH or MEDGEMMA_ID
print(f"source: {SRC}")
if MEDGEMMA_PATH is None:
    print("[!] Pulling from Hugging Face. medgemma-27b-text-it is ~54 GB in bf16 and")
    print("    the repo is GATED — accept the HAI-DEF terms and set HF_TOKEN in")
    print("    Kaggle Secrets first. Attaching it from Kaggle Models and setting")
    print("    MEDGEMMA_PATH in K0 avoids the download entirely.")
    try:
        from kaggle_secrets import UserSecretsClient
        os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
        print("    HF_TOKEN loaded from Kaggle Secrets.")
    except Exception as e:
        print(f"    (no HF_TOKEN in Secrets: {e})")

bnb_cfg = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=(torch.float16 if USE_FP16_COMPUTE else torch.float32),
)

t0 = time.time()
med_tok = AutoTokenizer.from_pretrained(SRC)
med_model = AutoModelForCausalLM.from_pretrained(
    SRC,
    quantization_config=bnb_cfg,
    device_map="auto",                      # pipeline-splits layers across both T4s
    attn_implementation="eager",            # no FlashAttention-2 on Turing
    dtype=(torch.float16 if USE_FP16_COMPUTE else torch.float32),
    low_cpu_mem_usage=True,
    max_memory={0: "13GiB", 1: "13GiB", "cpu": "0GiB"},  # fail loudly, never offload
).eval()
print(f"\nloaded in {time.time()-t0:.0f}s")
print("device map (first few):",
      dict(list(getattr(med_model, 'hf_device_map', {}).items())[:4]))
for i in range(torch.cuda.device_count()):
    print(f"  gpu{i} allocated {torch.cuda.memory_allocated(i)/2**30:.1f} GiB")

LOCAL[MEDGEMMA_ID] = {"tok": med_tok, "model": med_model, "source": SRC}

# ---- LANDMINE PROBE: does fp16 stay finite on a real prompt? ----------
_msgs = [{"role": "system", "content": RERANK_SYSTEM},
         {"role": "user",   "content": _p}]
try:
    _txt = med_tok.apply_chat_template(_msgs, add_generation_prompt=True, tokenize=False)
    _tmpl = "system+user"
except Exception:
    _txt = med_tok.apply_chat_template(
        [{"role": "user", "content": RERANK_SYSTEM + "\n\n" + _p}],
        add_generation_prompt=True, tokenize=False)
    _tmpl = "merged (template rejected a system turn)"
_enc = med_tok(_txt, return_tensors="pt", add_special_tokens=False).to(med_model.device)
with torch.no_grad():
    _logits = med_model(**_enc).logits

_finite = bool(torch.isfinite(_logits).all().item())
print(f"\nchat template : {_tmpl}")
print(f"prompt tokens : {_enc['input_ids'].shape[-1]}")
print(f"finite logits : {_finite}")
print(f"max |logit|   : {_logits.abs().max().item():.1f}")

MEDGEMMA_COMPUTE_DTYPE = "fp16" if USE_FP16_COMPUTE else "fp32"
if not _finite and USE_FP16_COMPUTE:
    print("\n[!] fp16 probe FAILED (non-finite logits). Reloading at fp32.")
    del med_model; LOCAL.pop(MEDGEMMA_ID, None)
    import gc as _gc; _gc.collect(); torch.cuda.empty_cache()
    bnb_cfg = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                 bnb_4bit_use_double_quant=True,
                                 bnb_4bit_compute_dtype=torch.float32)
    med_model = AutoModelForCausalLM.from_pretrained(
        SRC, quantization_config=bnb_cfg, device_map="auto",
        attn_implementation="eager", dtype=torch.float32,
        low_cpu_mem_usage=True,
        max_memory={0: "13GiB", 1: "13GiB", "cpu": "0GiB"}).eval()
    LOCAL[MEDGEMMA_ID] = {"tok": med_tok, "model": med_model, "source": SRC}
    MEDGEMMA_COMPUTE_DTYPE = "fp32"
    _enc = med_tok(_txt, return_tensors="pt", add_special_tokens=False).to(med_model.device)
    with torch.no_grad():
        _logits = med_model(**_enc).logits
    _finite = bool(torch.isfinite(_logits).all().item())
    print("fp32 finite logits:", _finite)
print(f"\ncompute dtype in use: {MEDGEMMA_COMPUTE_DTYPE}  <-- QUOTE IN METHODS")
if not _finite:
    raise RuntimeError(
        "fp16 overflow: logits contain inf/NaN. 4-bit MedGemma 27B on T4 is not usable "
        "as configured. Options, in order of preference: (a) try "
        "bnb_4bit_compute_dtype=torch.float32 (slower but numerically safe); "
        "(b) drop to MedGemma 4B at fp16 on one T4; (c) move to an Ampere GPU that "
        "supports bfloat16. Do NOT proceed — every downstream number would be noise.")
print(f"\nProbe passed at {MEDGEMMA_COMPUTE_DTYPE}.")

## B2d. Batched local generation + throughput probe

834 sequential MedGemma calls is roughly 10 hours. This cell measures the real batched throughput on your session so the planner can give you a number you can trust rather than an extrapolation.

In [ ]:
# =====================================================================
# B2d. Batched local generation. This is what makes 834 MedGemma calls fit
#      a Kaggle session: sequential generation amortises the 4-bit dequant
#      cost over one sequence, batched generation amortises it over `bs`.
#      Halves the batch automatically on OOM and remembers the size.
# =====================================================================
import gc

LOCAL_BATCH_STATE = {"size": None}


def _call_local_batch(model, prompts, max_new_tokens=None):
    """prompts: list of (system, user). Returns list of (text, n_tokens)."""
    tok, mdl = LOCAL[model]["tok"], LOCAL[model]["model"]
    mnt = max_new_tokens or LOCAL_MAX_NEW_TOKENS

    texts = []
    for sp, up in prompts:
        try:
            t = tok.apply_chat_template(
                [{"role": "system", "content": sp}, {"role": "user", "content": up}],
                add_generation_prompt=True, tokenize=False)
        except Exception:
            t = tok.apply_chat_template(
                [{"role": "user", "content": sp + "\n\n" + up}],
                add_generation_prompt=True, tokenize=False)
        texts.append(t)

    old_side = tok.padding_side
    tok.padding_side = "left"                 # decoder-only: pad on the LEFT
    if tok.pad_token_id is None:
        tok.pad_token = tok.eos_token
    try:
        enc = tok(texts, return_tensors="pt", padding=True,
                  add_special_tokens=False).to(mdl.device)
        n_in = enc["input_ids"].shape[-1]
        with torch.no_grad():
            out = mdl.generate(**enc, max_new_tokens=mnt, do_sample=False,
                               num_beams=1, pad_token_id=tok.pad_token_id)
    finally:
        tok.padding_side = old_side

    res = []
    for i in range(out.shape[0]):
        new = out[i][n_in:]
        n_real = int(enc["attention_mask"][i].sum())
        res.append((tok.decode(new, skip_special_tokens=True),
                    n_real + int(new.shape[-1])))
    return res


def local_batch_generate(model, prompts):
    bs = LOCAL_BATCH_STATE["size"] or LOCAL_BATCH
    out, i = [], 0
    while i < len(prompts):
        try:
            out.extend(_call_local_batch(model, prompts[i:i + bs]))
            i += bs
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache(); gc.collect()
            if bs == 1:
                raise
            bs = max(1, bs // 2)
            print(f"    [OOM] batch size -> {bs}")
    LOCAL_BATCH_STATE["size"] = bs
    return out


def rerank_batch(model, cases, cand_lists):
    """Uniform interface. Local models batch; API models loop (they are fast)."""
    prov, results = LIMITS[model]["provider"], []
    if prov == "local":
        prompts = [(RERANK_SYSTEM, build_prompt(c, ci))
                   for c, ci in zip(cases, cand_lists)]
        for (txt, ntok), ci in zip(local_batch_generate(model, prompts), cand_lists):
            USAGE[model]["calls"] += 1
            USAGE[model]["tokens"] += int(ntok)
            LAST_RAW[model] = txt
            try:
                order, nm = parse_ranking(txt, len(ci))
                results.append({"status": "ok",
                                "ranking": [int(ci[p]) for p in order],
                                "n_missing": nm})
            except Exception as e:
                results.append({"status": "error",
                                "error": f"{type(e).__name__}: {e}"[:300]})
    else:
        for c, ci in zip(cases, cand_lists):
            try:
                ranked, nm = rerank_one(model, c, ci)
                results.append({"status": "ok", "ranking": ranked, "n_missing": nm})
            except QuotaExhausted:
                raise
            except Exception as e:
                results.append({"status": "error",
                                "error": f"{type(e).__name__}: {e}"[:300]})
    return results


# ---- measure the real speedup on THIS session before committing hours -------
_probe_cases = [test_sets["T0"].iloc[i] for i in range(8)]
_probe_cands = [[int(x) for x in retrieve(c, EXT_RETRIEVER, top_k=TOP_K_RETRIEVE)[0]]
                for c in _probe_cases]

t0 = time.time()
_ = rerank_batch(MEDGEMMA_ID, _probe_cases[:1], _probe_cands[:1])
t_seq = time.time() - t0

LOCAL_BATCH_STATE["size"] = LOCAL_BATCH
t0 = time.time()
_r8 = rerank_batch(MEDGEMMA_ID, _probe_cases, _probe_cands)
t_bat = (time.time() - t0) / len(_probe_cases)
_nok = sum(1 for r in _r8 if r["status"] == "ok")

print(f"\nsequential : {t_seq:6.1f} s/call")
print(f"batched(bs={LOCAL_BATCH_STATE['size']}): {t_bat:6.1f} s/call   "
      f"speedup x{t_seq/max(t_bat,1e-9):.2f}")
print(f"parsed OK  : {_nok}/{len(_r8)}   <- must be {len(_r8)}/{len(_r8)}; if not, "
      f"left-padding or the chat template is wrong")

LOCAL_SEC_PER_CALL = t_bat
print(f"\n>>> projected MedGemma time, 834 external cases: "
      f"{834*t_bat/3600:.1f} h")


In [ ]:
# =====================================================================
# K0d. Run planner. Tells you, before you commit, whether the remaining
#      work fits your session and your weekly accelerator quota.
# =====================================================================
def plan(sec_per_local_call=None):
    spc = sec_per_local_call or globals().get("LOCAL_SEC_PER_CALL", 44.0)
    measured = "measured this session" if "LOCAL_SEC_PER_CALL" in globals() \
        else "ASSUMED (run the batching probe for a real number)"

    n_ext = 834
    done = 0
    if os.path.exists(OUT + "rarebench_rerank_v5.pkl"):
        with open(OUT + "rarebench_rerank_v5.pkl", "rb") as f:
            st = pickle.load(f)
        done = sum(1 for k, v in st.items()
                   if isinstance(k, tuple) and len(k) == 2 and k[0] != "cands"
                   and k[0] != "meta" and k[1] == MEDGEMMA_ID
                   and isinstance(v, dict) and v.get("status") == "ok")
    remaining = max(0, n_ext - done)

    rows = [
        ("K0-A5c  KB, embeddings, retrieval ablation", 0.5, "GPU, one-off"),
        ("D1      three phenotype baselines",          0.2, "CPU, vectorised"),
        ("B2b     load MedGemma 4-bit from HF",        0.2, "0.03 h if mounted from Kaggle Models"),
        ("B4      internal 400x3 re-rank",             0.0, "SKIP if v4 checkpoint present"),
        ("R0-R3   RareBench download, map, ablation",  0.3, "CPU"),
        (f"R4      external MedGemma ({remaining} left)",
         remaining * spc / 3600, f"{spc:.0f} s/call, {measured}"),
        ("R4      external Mistral (834)",             834 * 1.4 / 3600, "API, paced"),
        ("R5-R10  analysis, tables, figures",          0.2, "CPU"),
    ]
    total = sum(r[1] for r in rows)

    print(f"{'stage':46s}{'hours':>8s}   note")
    for nm, h, note in rows:
        print(f"{nm:46s}{h:>8.2f}   {note}")
    print(f"{'TOTAL':46s}{total:>8.2f}")

    print(f"\nKaggle documents 12 h per session and ~30 h/week of accelerator time.")
    print("VERIFY BOTH on your account — they change.")
    if total <= SESSION_BUDGET_H:
        print(f"\n>>> Fits one session ({total:.1f} h vs {SESSION_BUDGET_H} h budget).")
    else:
        n = int(np.ceil(total / SESSION_BUDGET_H))
        print(f"\n>>> Needs {n} sessions at {SESSION_BUDGET_H} h. The R4 runner stops")
        print(f">>> cleanly at the budget and resumes from the checkpoint, so just")
        print(f">>> re-run R4 in a fresh session {n-1} more time(s).")
    if total > 30:
        print(">>> WARNING: exceeds a typical weekly GPU quota. Reduce LOCAL_BATCH")
        print(">>> risk by mounting MedGemma from Kaggle Models, or drop to")
        print(">>> MedGemma 4B for a sensitivity check.")
    return total

_ = plan()


## B3. Borda / RRF fusion + disagreement metrics

In [ ]:
# =====================================================================
# B3. Fusion + disagreement
# =====================================================================
from scipy.stats import kendalltau

def borda_fuse(rank_lists, weights=None):
    weights = weights or [1.0] * len(rank_lists)
    pts = defaultdict(float)
    for lst, w in zip(rank_lists, weights):
        L = len(lst)
        for pos, item in enumerate(lst):
            pts[item] += w * (L - pos)
    return [i for i, _ in sorted(pts.items(), key=lambda x: -x[1])]


def rrf_fuse_lists(rank_lists, k=60, weights=None):
    weights = weights or [1.0] * len(rank_lists)
    pts = defaultdict(float)
    for lst, w in zip(rank_lists, weights):
        for pos, item in enumerate(lst):
            pts[item] += w / (k + pos + 1)
    return [i for i, _ in sorted(pts.items(), key=lambda x: -x[1])]


def _tau(x, y):
    r = kendalltau(x, y)
    return getattr(r, "statistic", None) if hasattr(r, "statistic") else r.correlation


def pairwise_disagreement(rank_lists):
    """(1 - tau)/2 for every pair. Returns a square matrix with NaN on the diagonal."""
    n = len(rank_lists)
    pos = [{it: i for i, it in enumerate(l)} for l in rank_lists]
    common = set(rank_lists[0])
    for p in pos[1:]:
        common &= set(p)
    common = sorted(common, key=lambda c: pos[0][c])
    M = np.full((n, n), np.nan)
    if len(common) < 3:
        return M
    for i in range(n):
        for j in range(i + 1, n):
            t = _tau([pos[i][c] for c in common], [pos[j][c] for c in common])
            if t is not None and not np.isnan(t):
                v = float(min(1.0, max(0.0, (1 - t) / 2)))
                M[i, j] = M[j, i] = v
    return M


def kendall_disagreement(rank_lists):
    """0.0 = models agree perfectly, 1.0 = perfectly reversed."""
    M = pairwise_disagreement(rank_lists)
    iu = np.triu_indices(len(rank_lists), k=1)
    vals = M[iu]
    vals = vals[~np.isnan(vals)]
    return float(np.mean(vals)) if len(vals) else 0.0


def topk_disagreement(rank_lists, k=5):
    sets = [set(l[:k]) for l in rank_lists]
    vals = []
    for i in range(len(sets)):
        for j in range(i + 1, len(sets)):
            u = len(sets[i] | sets[j])
            vals.append(1 - len(sets[i] & sets[j]) / u if u else 0.0)
    return float(np.mean(vals)) if vals else 0.0


_a = [1, 2, 3, 4, 5]; _b = [2, 1, 4, 3, 5]
assert kendall_disagreement([_a, _a]) < 1e-9, "identical rankings must score ~0"
assert kendall_disagreement([_a, _a[::-1]]) > 1 - 1e-9, "reversed rankings must score ~1"
assert borda_fuse([_a, _a])[0] == 1 and rrf_fuse_lists([_a, _a])[0] == 1
print("Fusion + disagreement OK.",
      f"example kendall={kendall_disagreement([_a, _b]):.3f}",
      f"top3-jaccard={topk_disagreement([_a, _b], k=3):.3f}")

## B4. Internal batch runner

**Skip this if you still have v4's `llm_rerank_checkpoint.pkl`** — the internal arm is unchanged and already complete at 400/400 for all three members. Re-running costs ~5 GPU-hours for identical numbers.

In [ ]:
import os

OUT = "/kaggle/working/"
os.makedirs(OUT, exist_ok=True)

print("OUT =", OUT)
print("exists =", os.path.isdir(OUT))

In [ ]:
# =====================================================================
# B4. Checkpointed, quota-aware, resumable batch runner
# =====================================================================
CKPT = OUT + "llm_rerank_checkpoint.pkl"
LIMIT = None          # set to 10 for a dry run, then None for the real thing


def load_ckpt():
    if os.path.exists(CKPT):
        with open(CKPT, "rb") as f:
            return pickle.load(f)
    return {}


def save_ckpt(d):
    tmp = CKPT + ".tmp"
    with open(tmp, "wb") as f:
        pickle.dump(d, f)
    os.replace(tmp, CKPT)          # atomic: a mid-write disconnect cannot corrupt it


def ckpt_counts(store):
    ok, err = defaultdict(int), defaultdict(int)
    for k, v in store.items():
        if isinstance(k, tuple) and len(k) == 3 and k[0] != "cands" and isinstance(v, dict):
            (ok if v.get("status") == "ok" else err)[k[2]] += 1
    return ok, err


def run_tier(tier, models=ENSEMBLE, retriever=None, limit=LIMIT, t_start=None):
    retriever = retriever or RETRIEVER
    store = load_ckpt()
    df = test_sets[tier]
    if limit:
        df = df.iloc[:limit]
    exhausted, n_done, t_start = set(), 0, t_start or time.time()

    for ci, (_, row) in enumerate(df.iterrows()):
        cand_idx, _ = retrieve(row, retriever, top_k=TOP_K_RETRIEVE)
        cand_idx = [int(x) for x in cand_idx]
        store.setdefault(("cands", tier, ci), cand_idx)

        for m in models:
            key = (tier, ci, m)
            if key in store or m in exhausted:
                continue
            try:
                ranked, nmiss = rerank_one(m, row, cand_idx)
                store[key] = {"status": "ok", "ranking": ranked, "n_missing": nmiss}
                n_done += 1
            except QuotaExhausted as e:
                print(f"\n  [QUOTA] {e}")
                print(f"  -> skipping {m} for the rest of this run; re-run to resume.")
                exhausted.add(m)
            except Exception as e:
                msg = f"{type(e).__name__}: {e}"[:300]
                store[key] = {"status": "error", "error": msg}
                print(f"  [ERR] {tier} c{ci} {m.split('/')[-1]}: {msg}")

        save_ckpt(store)                         # every case, not every 10

        if ci % 5 == 0 or ci == len(df) - 1:
            ok, err = ckpt_counts(store)
            rate = n_done / max(1e-9, time.time() - t_start)
            left = (len(df) - ci - 1) * len(models)
            eta  = left / rate / 60 if rate > 0 else float("nan")
            print(f"  {tier} {ci+1}/{len(df)} | "
                  + " ".join(f"{m.split('/')[-1][:12]}:{ok[m]}" for m in models)
                  + f" | err={sum(err.values())} | ETA(tier) {eta:.0f}m")

        if len(exhausted) == len(models):
            print("  All models exhausted — stopping. Re-run to resume.")
            break

    return store


# ---- RUN ---------------------------------------------------------------
_t0 = time.time()
print(f"retriever={RETRIEVER}  BEST_RRF={BEST_RRF}  limit={LIMIT}\n")
for tier in TIERS:
    print(f"=== {tier} ===")
    store = run_tier(tier, t_start=_t0)
    print(usage_report().to_string(index=False), "\n")

print(f"Total wall time: {(time.time()-_t0)/3600:.2f} h")

In [ ]:
# =====================================================================
# B4b. Completeness audit — read this before scoring
# =====================================================================
store = load_ckpt()
target = N_TEST
print(f"{'tier':6s}" + "".join(f"{m.split('/')[-1][:16]:>18s}" for m in ENSEMBLE))
short = []
for tier in TIERS:
    line = f"{tier:6s}"
    for m in ENSEMBLE:
        ok = sum(1 for k, v in store.items()
                 if isinstance(k, tuple) and len(k) == 3 and k[0] == tier and k[2] == m
                 and v.get("status") == "ok")
        er = sum(1 for k, v in store.items()
                 if isinstance(k, tuple) and len(k) == 3 and k[0] == tier and k[2] == m
                 and v.get("status") == "error")
        line += f"{f'{ok}/{target}' + (f' e{er}' if er else ''):>18s}"
        if ok < target:
            short.append((tier, m, target - ok))
    print(line)

if short:
    print("\nIncomplete:")
    for tier, m, n in short:
        print(f"  {tier} {m}: {n} missing")
    print("\nRe-run B4 — it resumes and re-pays for nothing already done.")
else:
    print(f"\nAll {len(ENSEMBLE)} x {len(TIERS)} x {target} = "
          f"{len(ENSEMBLE)*len(TIERS)*target} calls complete.")

errs = defaultdict(list)
for k, v in store.items():
    if isinstance(k, tuple) and len(k) == 3 and isinstance(v, dict) and v.get("status") == "error":
        errs[k[2]].append(v.get("error", "?"))
for m, e in errs.items():
    print(f"\n--- {m}: {len(e)} errors, distinct causes ---")
    for msg in list(dict.fromkeys(e))[:5]:
        print("   ", msg[:250])

In [ ]:
# =====================================================================
# B4c. Clear ONLY failed entries, so a re-run retries just those
# =====================================================================
store = load_ckpt()
dropped = 0
for k in list(store.keys()):
    if (isinstance(k, tuple) and len(k) == 3 and k[0] != "cands"
            and isinstance(store[k], dict) and store[k].get("status") == "error"):
        del store[k]; dropped += 1
save_ckpt(store)
kept = sum(1 for k, v in store.items()
           if isinstance(k, tuple) and len(k) == 3 and k[0] != "cands"
           and isinstance(v, dict) and v.get("status") == "ok")
print(f"Dropped {dropped} failed entries. Kept {kept} successful ones.")
print("Re-run B4 to retry only what failed.")

## B5. Paired scoring across systems and tiers (internal)

In [ ]:
# =====================================================================
# B5. Paired scoring
# =====================================================================
store = load_ckpt()

def collect(tier, models=ENSEMBLE):
    """Cases where ALL models returned a valid ranking."""
    df = test_sets[tier]
    rows = []
    for ci in range(len(df)):
        recs = [store.get((tier, ci, m)) for m in models]
        if any(r is None or r.get("status") != "ok" for r in recs):
            continue
        cands = store.get(("cands", tier, ci))
        if cands is None:
            continue
        rows.append({"ci": ci, "true": df.iloc[ci]["true_disease"],
                     "cands": cands, "rankings": [r["ranking"] for r in recs],
                     "n_missing": [r["n_missing"] for r in recs]})
    return rows


def hit_at(ranked_idx, true_name, k):
    return true_name in kb.iloc[ranked_idx[:k]]["disease_name"].tolist()


K_EVAL = (1, 3, 5, 10)
rows_out, per_case = [], {}

for tier in TIERS:
    data = collect(tier)
    if not data:
        print(f"{tier}: no complete cases yet — skipping")
        continue

    systems = {"retrieval": [], "fuse_borda": [], "fuse_rrf": [], "oracle_best": []}
    for i, m in enumerate(ENSEMBLE):
        systems[f"llm_{chr(65+i)}"] = []

    dis_kendall, dis_topk, truth, pair_mats = [], [], [], []

    for d in data:
        truth.append(d["true"])
        systems["retrieval"].append(d["cands"])
        for i, m in enumerate(ENSEMBLE):
            systems[f"llm_{chr(65+i)}"].append(d["rankings"][i])
        systems["fuse_borda"].append(borda_fuse(d["rankings"]))
        systems["fuse_rrf"].append(rrf_fuse_lists(d["rankings"]))
        best = min(d["rankings"],
                   key=lambda r: (kb.iloc[r]["disease_name"].tolist().index(d["true"])
                                  if d["true"] in kb.iloc[r]["disease_name"].tolist() else 999))
        systems["oracle_best"].append(best)
        M = pairwise_disagreement(d["rankings"])
        pair_mats.append(M)
        dis_kendall.append(kendall_disagreement(d["rankings"]))
        dis_topk.append(topk_disagreement(d["rankings"], k=5))

    per_case[tier] = {"truth": truth, "systems": systems,
                      "dis_kendall": dis_kendall, "dis_topk": dis_topk,
                      "pair_mats": pair_mats,
                      "n_missing": [d["n_missing"] for d in data]}

    for sname, ranklists in systems.items():
        rec = {f"R@{k}": float(np.mean([hit_at(r, t, k) for r, t in zip(ranklists, truth)]))
               for k in K_EVAL}
        rows_out.append({"tier": tier, "system": sname, "n": len(truth), **rec})

res = pd.DataFrame(rows_out)
if res.empty:
    print("\n[!] No completed cases in any tier. Run B4 first.")
else:
    res.to_csv(OUT + "table_llm_systems.csv", index=False)
    print(f"\nlegend: " + "  ".join(
        f"llm_{chr(65+i)}={m.split('/')[-1]}" for i, m in enumerate(ENSEMBLE)))
    print("\nRecall@5 by tier:")
    print(res.pivot(index="system", columns="tier", values="R@5").round(3).to_string())
    print("\nSaved table_llm_systems.csv")

In [ ]:
# =====================================================================
# B5b. Does the ensemble beat retrieval alone? (paired, per tier)
# =====================================================================
print("  ".join(f"llm_{chr(65+i)}={m.split('/')[-1]}" for i, m in enumerate(ENSEMBLE)))
print()
print(f"{'tier':6s}{'system':14s}{'R@5':>8s}{'vs retrieval':>14s}{'p':>10s}{'95% CI':>20s}")
for tier in TIERS:
    if tier not in per_case:
        continue
    pc = per_case[tier]
    base = [hit_at(r, t, 5) for r, t in zip(pc["systems"]["retrieval"], pc["truth"])]
    for sname in [f"llm_{chr(65+i)}" for i in range(len(ENSEMBLE))] + \
                 ["fuse_borda", "fuse_rrf", "oracle_best"]:
        if sname not in pc["systems"]:
            continue
        h = [hit_at(r, t, 5) for r, t in zip(pc["systems"][sname], pc["truth"])]
        p, b, c = mcnemar(h, base)
        reg_p("F2_internal_llm", f"{tier}: {sname} vs retrieval (R@5)", p,
              a=float(np.mean(h)), b=float(np.mean(base)), n=len(h))
        lo, hi = wilson(h)
        print(f"{tier:6s}{sname:14s}{np.mean(h):>8.3f}{np.mean(h)-np.mean(base):>+14.3f}"
              f"{p:>10.4f}{f'[{lo:.3f},{hi:.3f}]':>20s}")
    print()

print("Output-quality check (mean candidates the model failed to return):")
for tier in TIERS:
    if tier in per_case:
        nm = np.array(per_case[tier]["n_missing"])
        print(f"  {tier}: " + "  ".join(
            f"{m.split('/')[-1][:16]}={nm[:, i].mean():.2f}" for i, m in enumerate(ENSEMBLE)))
print("\n>>> A non-trivial mean here means parse_ranking is padding with retrieval order,")
print(">>> which inflates that model's score toward the retrieval baseline. Watch the")
print(">>> local model especially — it has no server-side JSON validator.")

## B5c. Lineage check: are MedGemma and Gemini too correlated to be independent voters?

This is the cell that defends your central claim. MedGemma is a Gemma-3 derivative and
Gemma shares Google's research lineage with Gemini, so a reviewer will ask whether low
disagreement between slots A and B reflects genuine confidence or merely shared inductive
bias. Your abstention signal only means what you claim it means if the voters are
substantially independent.

**How to read the output.** If `A-B` disagreement is markedly *lower* than `A-C` and
`B-C` across tiers, the Google-lineage pair is behaving as one voter and you should either
drop Gemini or report a two-voter ensemble. If the three pairs are comparable, you have a
published, quantitative answer to the objection.

Either result is a finding. The failure mode is not running the check.

In [ ]:
# =====================================================================
# B5c. Pairwise disagreement matrix + the diversity argument
# =====================================================================
import warnings

labels = [m.split("/")[-1] for m in ENSEMBLE]
n = len(ENSEMBLE)
pairs = [(i, j) for i in range(n) for j in range(i + 1, n)]

print("Mean pairwise Kendall disagreement, (1 - tau)/2, per tier")
print("0 = the two models rank identically, 1 = perfectly reversed\n")
hdr = f"{'tier':6s}" + "".join(
    f"{labels[i][:10]+'|'+labels[j][:10]:>24s}" for i, j in pairs)
print(hdr)

rows = []
for tier in TIERS:
    if tier not in per_case:
        continue
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", RuntimeWarning)   # all-NaN diagonal
        M = np.nanmean(np.stack(per_case[tier]["pair_mats"]), axis=0)
    line = f"{tier:6s}"
    for i, j in pairs:
        line += f"{M[i, j]:>24.3f}"
        rows.append({"tier": tier, "model_a": labels[i], "model_b": labels[j],
                     "mean_disagreement": round(float(M[i, j]), 4)})
    print(line)

pdf = pd.DataFrame(rows)
if pdf.empty:
    print("\n[!] No scored tiers yet.")
else:
    pdf.to_csv(OUT + "table_pairwise_disagreement.csv", index=False)
    piv = pdf.pivot_table(index=["model_a", "model_b"], values="mean_disagreement")
    print("\nAveraged over all tiers:")
    print(piv.round(3).to_string())

    ab = pdf[(pdf.model_a == labels[0]) & (pdf.model_b == labels[1])].mean_disagreement.mean()
    others = pdf[~((pdf.model_a == labels[0]) & (pdf.model_b == labels[1]))].mean_disagreement.mean()
    print(f"\n  Google-lineage pair (A-B) : {ab:.3f}")
    print(f"  cross-lineage pairs mean  : {others:.3f}")
    ratio = ab / others if others else float("nan")
    print(f"  ratio                     : {ratio:.2f}")
    print()
    if ratio < 0.75:
        print("  >>> A-B disagree substantially LESS than the cross-lineage pairs.")
        print("  >>> Treat slots A and B as partially redundant. Either drop Gemini and")
        print("  >>> report a two-voter ensemble, or state this ratio explicitly and")
        print("  >>> argue why the medical continued-pretraining still leaves them")
        print("  >>> independent enough. Do not leave it unaddressed.")
    else:
        print("  >>> A-B disagreement is comparable to the cross-lineage pairs. The")
        print("  >>> shared Gemma/Gemini lineage does not appear to collapse them into")
        print("  >>> one voter on this task. Report this number — it is your answer to")
        print("  >>> the obvious reviewer objection.")

## B6. Risk-coverage: the central result

In [ ]:
# =====================================================================
# B6. Risk-coverage curves + AURC vs random abstention
# =====================================================================
def risk_coverage(hits, scores):
    """Ascending disagreement -> answer the most-confident first."""
    order = np.argsort(np.asarray(scores))
    h = np.asarray(hits, dtype=float)[order]
    cov = np.arange(1, len(h) + 1) / len(h)
    acc = np.cumsum(h) / np.arange(1, len(h) + 1)
    return cov, acc


def aurc(cov, acc):
    return float(np.trapezoid(acc, cov)) if hasattr(np, "trapezoid") else float(np.trapz(acc, cov))


fig, axes = plt.subplots(1, len(TIERS), figsize=(4.2 * len(TIERS), 4), sharey=True)
if len(TIERS) == 1:
    axes = [axes]

summary = []
for ax, tier in zip(axes, TIERS):
    if tier not in per_case:
        ax.set_title(f"{tier} (no data)"); continue
    pc = per_case[tier]
    hits = [hit_at(r, t, 5) for r, t in zip(pc["systems"]["fuse_rrf"], pc["truth"])]

    for scores, lab in ((pc["dis_kendall"], "Kendall disagreement"),
                        (pc["dis_topk"], "Top-5 Jaccard disagreement")):
        cov, acc = risk_coverage(hits, scores)
        ax.plot(cov, acc, label=lab)
        summary.append({"tier": tier, "signal": lab, "AURC": round(aurc(cov, acc), 4)})

    rng_a = np.random.RandomState(0)
    rand = np.mean([aurc(*risk_coverage(hits, rng_a.rand(len(hits)))) for _ in range(50)])
    ax.axhline(np.mean(hits), ls="--", c="gray", lw=1,
               label=f"full coverage ({np.mean(hits):.2f})")
    summary.append({"tier": tier, "signal": "random abstention", "AURC": round(rand, 4)})

    ax.set_title(tier); ax.set_xlabel("coverage"); ax.grid(alpha=.3); ax.set_ylim(0, 1)
axes[0].set_ylabel("Recall@5 among answered cases")
axes[0].legend(fontsize=7)
plt.tight_layout()
plt.savefig(OUT + "fig_risk_coverage.png", dpi=200, bbox_inches="tight")
plt.show()

sdf = pd.DataFrame(summary)
if sdf.empty:
    print("[!] No scored tiers yet — run B4 and B5 first.")
else:
    sdf.to_csv(OUT + "table_aurc.csv", index=False)
    print(sdf.pivot(index="signal", columns="tier", values="AURC").to_string())
    print("\nIf the disagreement rows are not clearly above 'random abstention',")
    print("the signal is weak — report that honestly, it is still a publishable finding.")

In [ ]:
# =====================================================================
# B7. Operating points — the table that goes in the paper
# =====================================================================
rows = []
for tier in TIERS:
    if tier not in per_case:
        continue
    pc = per_case[tier]
    hits = np.array([hit_at(r, t, 5) for r, t in zip(pc["systems"]["fuse_rrf"], pc["truth"])],
                    dtype=float)
    scores = np.array(pc["dis_kendall"])
    order = np.argsort(scores)
    for target_cov in (1.0, 0.8, 0.6, 0.4):
        n = max(1, int(round(len(hits) * target_cov)))
        sel = order[:n]
        lo, hi = wilson(hits[sel].astype(bool).tolist())
        rows.append({"tier": tier, "coverage": target_cov, "n_answered": n,
                     "Recall@5": round(float(hits[sel].mean()), 3),
                     "CI_low": round(lo, 3), "CI_high": round(hi, 3),
                     "abstain_thresh": round(float(scores[order][n-1]), 3)})

op = pd.DataFrame(rows)
if op.empty:
    print("[!] Nothing scored yet.")
else:
    op.to_csv(OUT + "table_operating_points.csv", index=False)
    print(op.to_string(index=False))

## B8. Export + the provenance block for your Methods section

In [ ]:
# =====================================================================
# B8. Export everything, including exact versions and quant config
# =====================================================================
import platform

def _g(name, default=None):
    return globals().get(name, default)

try:
    import transformers as _tf, bitsandbytes as _bnb, accelerate as _acc
    _vers = {"transformers": _tf.__version__, "bitsandbytes": _bnb.__version__,
             "accelerate": _acc.__version__, "torch": torch.__version__,
             "python": platform.python_version()}
except Exception as e:
    _vers = {"error": str(e)}

provenance = {
    "config": {
        "n_test": N_TEST, "n_dev": int(len(dev_idx)), "top_k": TOP_K_RETRIEVE,
        "tiers": TIERS, "seed": SEED, "ensemble": ENSEMBLE,
        "retriever": RETRIEVER, "best_rrf": BEST_RRF,
        "kb_size": int(len(kb)), "have_ontology": HAVE_ONTOLOGY,
        "max_completion_tokens": MAX_COMPLETION_TOKENS,
        "local_max_new_tokens": LOCAL_MAX_NEW_TOKENS,
    },
    "local_model": {
        "id": MEDGEMMA_ID,
        "source": LOCAL.get(MEDGEMMA_ID, {}).get("source"),
        "quantization": "4-bit NF4, double_quant=True, compute_dtype=float16",
        "attn_implementation": "eager",
        "generation": "do_sample=False, num_beams=1 (greedy, deterministic)",
        "note": ("T4/Turing has no bfloat16 and no FlashAttention-2. This is a "
                 "4-bit fp16 result and is NOT numerically identical to the "
                 "released bf16 checkpoint. State this in the paper."),
    },
    "versions": _vers,
    "gpus": [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())],
    "usage": {m: dict(u) for m, u in USAGE.items()},
    "retrieval_ablation": _g("retrieval_results"),
    "llm_systems": (_g("res").to_dict("records")
                    if _g("res") is not None and not _g("res").empty else None),
}

with open(OUT + "final_results_v4.pkl", "wb") as f:
    pickle.dump(provenance, f)
with open(OUT + "provenance_v4.json", "w") as f:
    json.dump({k: v for k, v in provenance.items() if k != "retrieval_ablation"},
              f, indent=2, default=str)

print("Artefacts in", OUT)
for fn in ["disease_kb_v3.pkl", "test_sets_tiered.pkl", "dev_sets_tiered.pkl",
           "best_rrf.json", "table_rrf_tuning_dev.csv", "retrieval_ablation.pkl",
           "table_retrieval_ablation.csv", "table_llm_systems.csv",
           "table_pairwise_disagreement.csv", "table_aurc.csv",
           "table_operating_points.csv", "fig_degradation_retrieval.png",
           "fig_risk_coverage.png", "llm_rerank_checkpoint.pkl",
           "final_results_v4.pkl", "provenance_v4.json"]:
    print(f"  {'OK ' if os.path.exists(OUT + fn) else '-- '}{fn}")

print("\nFinal usage:")
print(usage_report().to_string(index=False))
print("\nprovenance_v4.json contains the exact versions and quant config. Paste it")
print("into your Methods section — a 4-bit local result is not reproducible without it.")

# Part C — external validation on RareBench

Chen et al., KDD '24 (arXiv:2402.06341). Real patient phenotype sets with confirmed diagnoses. This is the only part of the notebook where the queries were not manufactured from the index.

In [ ]:
# =====================================================================
# R0b. RareBench via direct file download (datasets>=4 dropped scripts).
#      Layout from the repo's own loader: data.zip -> data/{NAME}.jsonl
#      Fields: Phenotype[list[str]], RareDisease[list[str]], Department(str)
# =====================================================================
import zipfile
from collections import Counter
from huggingface_hub import hf_hub_download

REPO    = "chenxz/RareBench"
SUBSETS = ["RAMEDIS", "MME", "HMS", "LIRICAL", "PUMCH_ADM"]
RB_DIR  = OUT + "rarebench/"
os.makedirs(RB_DIR, exist_ok=True)

# ---- 1. pull data.zip + the disease code mapping ----------------------
zpath = hf_hub_download(REPO, "data.zip", repo_type="dataset",
                        local_dir=RB_DIR)
print("data.zip ->", zpath, f"({os.path.getsize(zpath)/1e6:.1f} MB)")

with zipfile.ZipFile(zpath) as z:
    names = z.namelist()
    print("\nzip contents:")
    for n in names:
        print("   ", n)
    z.extractall(RB_DIR)

dmap_path = hf_hub_download(REPO, "mapping/disease_mapping.json",
                            repo_type="dataset", local_dir=RB_DIR)
with open(dmap_path, encoding="utf-8") as f:
    DISEASE_MAP = json.load(f)
print(f"\ndisease_mapping.json: {len(DISEASE_MAP)} entries")
for k, v in list(DISEASE_MAP.items())[:8]:
    print(f"    {k!r} -> {str(v)[:70]!r}")
print("  key prefixes:",
      dict(Counter(k.split(":")[0] if ":" in k else "NOPREFIX"
                   for k in DISEASE_MAP).most_common(10)))

# ---- 2. read the jsonl files -----------------------------------------
def _find_jsonl(name):
    for root, _, fs in os.walk(RB_DIR):
        for fn in fs:
            if fn == f"{name}.jsonl":
                return os.path.join(root, fn)
    return None

RB = {}
for name in SUBSETS:
    p = _find_jsonl(name)
    if p is None:
        print(f"\n=== {name}: NOT FOUND in extracted zip")
        continue
    rows = []
    with open(p, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    RB[name] = rows
    print(f"\n=== {name}: {len(rows)} rows | keys = {sorted(rows[0].keys())}")
    for ex in rows[:2]:
        for k, v in ex.items():
            s = str(v)
            print(f"    {k}: {s[:200]}{' ...' if len(s) > 200 else ''}")
        print("    ---")

# ---- 3. code formats + how well do labels hit OUR KB? -----------------
kb_orpha = set(kb["orpha_id"].astype(str))
print(f"\nKB: {len(kb_orpha)} diseases | id examples {sorted(kb_orpha)[:5]}")

def _codes(v):
    return [str(x) for x in (v if isinstance(v, list) else [v]) if x is not None]

for name, rows in RB.items():
    dis_pref, phe_pref = Counter(), Counter()
    n_dis, n_phe = [], []
    hit_raw = hit_strip = hit_orpha_only = 0
    in_dmap = dmap_tot = 0
    phe_ok = phe_tot = 0
    for ex in rows:
        ds = _codes(ex.get("RareDisease"))
        ps = _codes(ex.get("Phenotype"))
        n_dis.append(len(ds)); n_phe.append(len(ps))
        for c in ds:
            dis_pref[c.split(":")[0] if ":" in c else "NOPREFIX"] += 1
            dmap_tot += 1
            if c in DISEASE_MAP:
                in_dmap += 1
        for p in ps:
            phe_pref[p.split(":")[0] if ":" in p else "NOPREFIX"] += 1
        if any(c in kb_orpha for c in ds):
            hit_raw += 1
        if any(c.split(":")[-1] in kb_orpha for c in ds):
            hit_strip += 1
        if any(c.upper().startswith("ORPHA") and c.split(":")[-1] in kb_orpha
               for c in ds):
            hit_orpha_only += 1
        for p in ps:
            phe_tot += 1
            try:
                if canon(p) in HPO:
                    phe_ok += 1
            except Exception:
                pass
    n = len(rows)
    print(f"\n--- {name}  n={n}")
    print(f"    diseases/case   mean={np.mean(n_dis):.2f} max={max(n_dis)}")
    print(f"    phenotypes/case mean={np.mean(n_phe):.1f} "
          f"min={min(n_phe)} max={max(n_phe)}")
    print(f"    disease prefixes  : {dict(dis_pref.most_common(8))}")
    print(f"    phenotype prefixes: {dict(phe_pref.most_common(4))}")
    print(f"    codes found in disease_mapping.json: {in_dmap}/{dmap_tot}")
    print(f"    case label in OUR KB: raw={hit_raw}/{n}  "
          f"strip-prefix={hit_strip}/{n}  ORPHA-only={hit_orpha_only}/{n}")
    print(f"    HPO codes resolvable in our closure: {phe_ok}/{phe_tot}"
          f" ({100*phe_ok/max(1,phe_tot):.0f}%)")

In [ ]:
# =====================================================================
# R1. Map RareBench cases -> our Orphanet KB. Coverage audit only.
#     ORPHA-prefixed codes ONLY: bare prefix-stripping produced 4 false
#     matches on RAMEDIS (595 vs 591) where an OMIM/CCRD number collides
#     with an unrelated OrphaCode.
# =====================================================================
RB_SUBSETS = [s for s in ["RAMEDIS", "MME", "HMS", "LIRICAL"] if s in RB]

# KB lookup: orpha_id -> row indices (a code may map to >1 row)
kb_by_orpha = defaultdict(list)
for i, oid in enumerate(kb["orpha_id"].astype(str)):
    kb_by_orpha[oid].append(i)
dupe_codes = {k: v for k, v in kb_by_orpha.items() if len(v) > 1}
print(f"KB: {len(kb_by_orpha)} distinct orpha_ids, "
      f"{len(dupe_codes)} appearing on >1 row")

def norm_code(c):
    c = str(c).strip().upper()
    return re.sub(r"\s*:\s*", ":", c)

def orpha_ids(codes):
    out = []
    for c in codes:
        c = norm_code(c)
        if c.startswith("ORPHA:"):
            out.append(c.split(":", 1)[1].strip())
    return out

RB_MAPPED, audit = {}, []
for name in RB_SUBSETS:
    rows, kept = RB[name], []
    n_no_orpha = n_orpha_miss = n_no_phe = 0
    seen_sig = {}
    n_dupe = 0
    for ri, ex in enumerate(rows):
        codes = [str(x) for x in (ex.get("RareDisease") or [])]
        phes  = [str(x) for x in (ex.get("Phenotype") or [])]
        oids  = orpha_ids(codes)
        if not phes:
            n_no_phe += 1; continue
        if not oids:
            n_no_orpha += 1; continue
        gold = sorted({i for o in oids for i in kb_by_orpha.get(o, [])})
        if not gold:
            n_orpha_miss += 1; continue
        sig = (tuple(sorted(set(phes))), tuple(gold))
        if sig in seen_sig:
            n_dupe += 1; continue
        seen_sig[sig] = ri
        kept.append({
            "subset":       name,
            "row":          ri,
            "case_id":      f"{name}_{ri}",
            "hpo_codes":    phes,
            "orpha_codes":  oids,
            "gold_kb_idx":  gold,
            "gold_names":   [kb.iloc[i]["disease_name"] for i in gold],
            "n_phe":        len(phes),
        })
    RB_MAPPED[name] = kept
    audit.append(dict(subset=name, total=len(rows), kept=len(kept),
                      no_phenotype=n_no_phe, no_orpha_code=n_no_orpha,
                      orpha_not_in_kb=n_orpha_miss, exact_duplicate=n_dupe))

aud = pd.DataFrame(audit)
aud["kept_pct"] = (100 * aud["kept"] / aud["total"]).round(1)
print("\nCoverage audit (this table goes in the paper):")
print(aud.to_string(index=False))
print(f"\nTOTAL kept: {aud['kept'].sum()} of {aud['total'].sum()} cases")

# ---- shape of what survived ------------------------------------------
allc = [c for v in RB_MAPPED.values() for c in v]
ngold = Counter(len(c["gold_kb_idx"]) for c in allc)
print(f"\ngold diseases per case: {dict(sorted(ngold.items()))}")
print("  (>1 means the case maps to several KB rows; scoring counts a hit "
      "if ANY gold disease is retrieved)")
nph = np.array([c["n_phe"] for c in allc])
print(f"phenotypes/case: mean={nph.mean():.1f} median={np.median(nph):.0f} "
      f"min={nph.min()} max={nph.max()}")
print("\nOur own test set for comparison:")
for t in TIERS:
    print(f"  {t}: mean {test_sets[t].query_symptoms.apply(len).mean():.1f} terms")

print("\nSample mapped cases:")
for c in allc[:5]:
    print(f"  {c['case_id']}: {len(c['hpo_codes'])} HPO -> "
          f"ORPHA {c['orpha_codes']} -> {c['gold_names'][:2]}")

# ---- what we are DROPPING: is it systematic? -------------------------
#      OMIM-only cases are diseases with no Orphanet equivalent in our KB.
#      Report this honestly rather than quietly excluding them.
lost_names = []
for name in RB_SUBSETS:
    for ex in RB[name]:
        codes = [norm_code(x) for x in (ex.get("RareDisease") or [])]
        if not orpha_ids(codes):
            nm = next((DISEASE_MAP.get(c) for c in codes if c in DISEASE_MAP), None)
            if nm:
                lost_names.append(nm.split("/")[0][:60])
print(f"\nDropped-for-no-ORPHA-code, most frequent diagnoses ({len(lost_names)} total):")
for nm, k in Counter(lost_names).most_common(10):
    print(f"    {k:3d}  {nm}")

with open(OUT + "rarebench_mapped.pkl", "wb") as f:
    pickle.dump(RB_MAPPED, f)
aud.to_csv(OUT + "table_rarebench_coverage.csv", index=False)
print("\nSaved rarebench_mapped.pkl + table_rarebench_coverage.csv")

In [ ]:
# =====================================================================
# R2. Build RareBench case frames in the shape retrieve() expects.
#     Same query_text template as A3 so nothing downstream changes.
# =====================================================================
with open(OUT + "rarebench_mapped.pkl", "rb") as f:
    RB_MAPPED = pickle.load(f)

rows, unresolved = [], Counter()
for name, cases in RB_MAPPED.items():
    for c in cases:
        ids, names = [], []
        seen = set()
        for h in c["hpo_codes"]:
            cid = canon(h)                       # collapses alt_ids
            if cid in seen:
                continue
            seen.add(cid)
            if cid in HPO and HPO[cid].get("name"):
                ids.append(cid); names.append(HPO[cid]["name"])
            else:
                unresolved[h] += 1
        if len(ids) < 2:                         # need >=2 usable terms
            continue
        rows.append({
            "case_id":        c["case_id"],
            "subset":         c["subset"],
            "true_disease":   c["gold_names"][0],
            "gold_kb_idx":    c["gold_kb_idx"],
            "single_gold":    len(c["gold_kb_idx"]) == 1,
            "orpha_codes":    c["orpha_codes"],
            "query_hpo_ids":  ids,
            "query_symptoms": names,
            "query_text":     "Patient presents with: " + ", ".join(names),
            "n_phe":          len(ids),
        })

rb_cases = pd.DataFrame(rows).reset_index(drop=True)

# length bins, so R3 can report a slice comparable to our own tiers
def _bin(n):
    if n <= 6:   return "1-6"
    if n <= 9:   return "7-9"
    if n <= 14:  return "10-14"
    return "15+"
rb_cases["len_bin"] = rb_cases["n_phe"].apply(_bin)

print(f"RareBench case frame: {len(rb_cases)} cases")
if unresolved:
    print(f"[!] {sum(unresolved.values())} HPO codes unresolved after canon(): "
          f"{list(unresolved.items())[:5]}")
else:
    print("All HPO codes resolved to named terms.")

print("\nby subset:")
print(rb_cases.groupby("subset")
      .agg(n=("case_id", "size"),
           single_gold=("single_gold", "sum"),
           mean_phe=("n_phe", "mean")).round(1).to_string())

print("\nby query length (for the length-matched comparison in R3):")
print(rb_cases["len_bin"].value_counts()
      .reindex(["1-6", "7-9", "10-14", "15+"]).to_string())
print("\n  our tiers sit at: T1 3.1  T3 4.5  T0 5.8  T2 8.3 terms")

# duplicate gold check across subsets (same disease many times?)
top = Counter(rb_cases["true_disease"]).most_common(8)
print("\nmost frequent gold diagnoses:")
for nm, k in top:
    print(f"    {k:3d}  {nm[:60]}")
print(f"  distinct diseases: {rb_cases['true_disease'].nunique()} "
      f"across {len(rb_cases)} cases")

# ---- smoke: does retrieve() work unchanged on these? -----------------
print("\n--- retrieval smoke test (3 cases, hybrid_tuned) ---")
for _, c in rb_cases.head(3).iterrows():
    idx, sc = retrieve(c, RETRIEVER, top_k=25)
    hit = [r for r in range(len(idx)) if idx[r] in c["gold_kb_idx"]]
    print(f"  {c['case_id']:14s} n_phe={c['n_phe']:2d} "
          f"rank={hit[0]+1 if hit else '>25'}  gold={c['true_disease'][:40]}")
    print(f"      top1: {kb.iloc[idx[0]]['disease_name'][:55]}")

rb_cases.to_pickle(OUT + "rarebench_cases.pkl")
print(f"\nSaved rarebench_cases.pkl ({len(rb_cases)} cases)")

In [ ]:
# =====================================================================
# R2b. DIAGNOSTIC — why did the smoke test miss? Bug or distribution shift?
#      Nothing is scored or saved here.
# =====================================================================
rb_cases = pd.read_pickle(OUT + "rarebench_cases.pkl")

def signal_ranks(case, gold_idx):
    """Rank of the best gold disease under each signal separately."""
    qt, qids = case["query_text"], case["query_hpo_ids"]
    sigs = {
        "bm25":         bm25_scores(qt),
        "ic":           ic_scores(qids),
        "dense":        dense_scores(qt, "sapbert", "noname"),
        "hybrid_tuned": rrf(signal_scores(case), k=BEST_RRF["k"],
                            weights=BEST_RRF["weights"]),
    }
    out = {}
    for k, s in sigs.items():
        order = np.argsort(-np.asarray(s))
        pos = {int(v): r for r, v in enumerate(order)}
        out[k] = min(pos[g] + 1 for g in gold_idx)
    return out

print("=" * 70)
print("PART 1 — anatomy of the three failing cases")
print("=" * 70)
for _, c in rb_cases.head(3).iterrows():
    g = c["gold_kb_idx"][0]
    krow = kb.iloc[g]
    q_ids = set(canon(h) for h in c["query_hpo_ids"])
    k_ids = set(canon(h) for h in krow["hpo_ids"])
    q_anc = set().union(*[ancestors_of(h) for h in c["query_hpo_ids"]]) if len(c["query_hpo_ids"]) else set()
    k_anc = set().union(*[ancestors_of(h) for h in krow["hpo_ids"]])
    print(f"\n--- {c['case_id']}  gold = {c['true_disease']}")
    print(f"  QUERY terms ({len(c['query_symptoms'])}):")
    for h, n in zip(c["query_hpo_ids"], c["query_symptoms"]):
        print(f"      {h}  {n}")
    print(f"  GOLD KB annotations ({krow['n_symptoms']}), first 12:")
    for h, n in list(zip(krow["hpo_ids"], krow["symptoms"]))[:12]:
        print(f"      {h}  {n}")
    print(f"  exact HPO overlap        : {len(q_ids & k_ids)} / {len(q_ids)}")
    print(f"  ancestor-closure overlap : {len(q_anc & k_anc)} "
          f"(query closure {len(q_anc)}, gold closure {len(k_anc)})")
    toks_q = set(tokenize(c["query_text"]))
    toks_k = set(tokenize(krow["text_no_name"]))
    print(f"  BM25 token overlap       : {len(toks_q & toks_k)} / {len(toks_q)}"
          f"  shared={sorted(toks_q & toks_k)[:8]}")
    print(f"  gold rank by signal      : {signal_ranks(c, c['gold_kb_idx'])}")

print("\n" + "=" * 70)
print("PART 2 — rank distribution over a 150-case sample (is it systematic?)")
print("=" * 70)
samp = rb_cases.sample(n=min(150, len(rb_cases)), random_state=SEED)
recs = []
for _, c in samp.iterrows():
    r = signal_ranks(c, c["gold_kb_idx"])
    r.update(subset=c["subset"], n_phe=c["n_phe"])
    recs.append(r)
rd = pd.DataFrame(recs)
for m in ["bm25", "ic", "dense", "hybrid_tuned"]:
    print(f"  {m:14s} R@1={np.mean(rd[m]<=1):.3f} R@5={np.mean(rd[m]<=5):.3f} "
          f"R@25={np.mean(rd[m]<=25):.3f} median_rank={rd[m].median():.0f}")
print("\n  hybrid_tuned R@25 by subset:")
print(rd.groupby("subset")["hybrid_tuned"]
      .agg(n="size", R25=lambda s: np.mean(s <= 25)).round(3).to_string())
print("\n  hybrid_tuned R@25 by query length:")
rd["len_bin"] = rd["n_phe"].apply(lambda n: "1-6" if n <= 6 else "7-9" if n <= 9
                                  else "10-14" if n <= 14 else "15+")
print(rd.groupby("len_bin")["hybrid_tuned"]
      .agg(n="size", R25=lambda s: np.mean(s <= 25)).round(3).to_string())

print("\n" + "=" * 70)
print("PART 3 — THE BUG TEST. Query built from the gold's OWN annotations.")
print("         These MUST come back at rank 1. If not, the fault is ours.")
print("=" * 70)
ok = 0
for _, c in rb_cases.head(6).iterrows():
    g = c["gold_kb_idx"][0]
    krow = kb.iloc[g]
    n = min(6, len(krow["hpo_ids"]))
    synth = {
        "query_hpo_ids":  list(krow["hpo_ids"])[:n],
        "query_symptoms": list(krow["symptoms"])[:n],
        "query_text": "Patient presents with: " + ", ".join(list(krow["symptoms"])[:n]),
    }
    idx, _ = retrieve(synth, RETRIEVER, top_k=25)
    rank = next((r + 1 for r, v in enumerate(idx) if int(v) == g), ">25")
    ok += (rank == 1)
    print(f"  {c['case_id']:14s} synthetic-from-gold rank = {rank:>4}  "
          f"{krow['disease_name'][:45]}")
print(f"\n  {ok}/6 at rank 1 -> "
      f"{'pipeline OK, failures are real' if ok >= 5 else 'PIPELINE BUG — stop and fix'}")

In [ ]:
# =====================================================================
# R2c. Corrected control. Sample terms the way A3 does (frequency-
#      weighted random, not top-k) on the ACTUAL RareBench gold diseases.
# =====================================================================
rng = np.random.RandomState(SEED)
gold_idx_all = sorted({g for gs in rb_cases["gold_kb_idx"] for g in gs})
print(f"{len(gold_idx_all)} distinct RareBench gold diseases in KB")

pick = rng.choice(gold_idx_all, size=min(40, len(gold_idx_all)), replace=False)
ranks = []
for g in pick:
    krow = kb.iloc[int(g)]
    ids, names = sample_true_terms(krow, rng)          # A3's own sampler
    synth = {"query_hpo_ids": ids, "query_symptoms": names,
             "query_text": "Patient presents with: " + ", ".join(names)}
    idx, _ = retrieve(synth, RETRIEVER, top_k=25)
    r = next((k + 1 for k, v in enumerate(idx) if int(v) == int(g)), 999)
    ranks.append(r)
ranks = np.array(ranks)
print(f"\nSynthetic-from-gold, A3 sampling, n={len(ranks)}:")
print(f"  R@1={np.mean(ranks<=1):.2f}  R@5={np.mean(ranks<=5):.2f}  "
      f"R@25={np.mean(ranks<=25):.2f}  median={np.median(ranks):.0f}")
print(f"  our T0 test set for reference: R@1=0.85 R@5=0.97 R@25=0.99")
print("\n  -> comparable to T0 means the pipeline is sound on these diseases,")
print("     and the RareBench gap is a property of the QUERIES, not the index.")

# term-level: how many RareBench query terms exist anywhere in the KB?
kb_terms = set()
for ids in kb["hpo_ids"]:
    kb_terms |= {canon(h) for h in ids}
kb_closure = set()
for ids in kb["hpo_ids"]:
    for h in ids:
        kb_closure |= ancestors_of(h)

rows = []
for _, c in rb_cases.iterrows():
    q = [canon(h) for h in c["query_hpo_ids"]]
    rows.append(dict(subset=c["subset"],
                     frac_in_kb=np.mean([t in kb_terms for t in q]),
                     frac_in_closure=np.mean([t in kb_closure for t in q])))
tt = pd.DataFrame(rows)
print("\nRareBench query terms present in KB annotations:")
print(tt.groupby("subset")[["frac_in_kb", "frac_in_closure"]].mean().round(3).to_string())
print(f"  overall exact: {tt.frac_in_kb.mean():.3f}   "
      f"via ancestor closure: {tt.frac_in_closure.mean():.3f}")

# which specific terms are most common and most often absent?
miss = Counter()
for _, c in rb_cases.iterrows():
    for h, n in zip(c["query_hpo_ids"], c["query_symptoms"]):
        if canon(h) not in kb_terms:
            miss[n] += 1
print("\nMost frequent RareBench terms ABSENT from every KB annotation:")
for n, k in miss.most_common(12):
    print(f"    {k:4d}  {n}")

## R3. Retrieval-only ablation on RareBench

Now includes the three phenotype-driven baselines, and every McNemar test registers into the correction registry.

In [ ]:
# =====================================================================
# R3. Retrieval-only ablation on RareBench (external validation).
#     No LLM calls. Pre-specified METHODS only — NO re-tuning here.
# =====================================================================
rb_cases = pd.read_pickle(OUT + "rarebench_cases.pkl")
t0 = time.time()

def _rank_of(scores, gold):
    order = np.argsort(-np.asarray(scores))
    pos = np.empty(len(order), dtype=np.int64)
    pos[order] = np.arange(len(order))
    return int(min(pos[g] for g in gold)) + 1

rank_rows = []
for n, (_, c) in enumerate(rb_cases.iterrows()):
    qt, qids, gold = c["query_text"], c["query_hpo_ids"], c["gold_kb_idx"]
    d_no   = dense_scores(qt, "sapbert", "noname")
    d_name = dense_scores(qt, "sapbert", "withname")
    s_bm   = bm25_scores(qt)
    s_ic   = ic_scores(qids)
    per = {m: BASELINE_SCORERS[m](c) for m in BASELINE_METHODS}
    per.update({
        "sapbert":           d_no,
        "sapbert_name":      d_name,
        "bm25":              s_bm,
        "ic":                s_ic,
        "hybrid_dense_bm25": rrf([d_no, s_bm]),
        "hybrid_dense_ic":   rrf([d_no, s_ic]),
        "hybrid":            rrf([d_no, s_bm, s_ic]),
        "hybrid_tuned":      rrf([d_no, s_bm, s_ic], k=BEST_RRF["k"],
                                 weights=BEST_RRF["weights"]),
    })
    r = {m: _rank_of(s, gold) for m, s in per.items()}
    r.update(case_id=c["case_id"], subset=c["subset"],
             len_bin=c["len_bin"], single_gold=c["single_gold"])
    rank_rows.append(r)
    if n % 200 == 0:
        print(f"  {n}/{len(rb_cases)} ... {time.time()-t0:.0f}s")

RB_RANKS = pd.DataFrame(rank_rows)
RB_RANKS.to_csv(OUT + "rarebench_ranks.csv", index=False)
print(f"done in {time.time()-t0:.0f}s\n")

METH = METHODS_V5

def boot_ci(hits, B=2000, seed=SEED):
    hits = np.asarray(hits, dtype=float)
    r = np.random.RandomState(seed)
    bs = [hits[r.randint(0, len(hits), len(hits))].mean() for _ in range(B)]
    return np.percentile(bs, 2.5), np.percentile(bs, 97.5)

def mcnemar_exact(a_hits, b_hits):
    a, b = np.asarray(a_hits), np.asarray(b_hits)
    n01 = int(np.sum(~a & b)); n10 = int(np.sum(a & ~b))
    n = n01 + n10
    if n == 0:
        return 1.0, n10, n01
    k = min(n01, n10)
    p = sum(math.comb(n, i) for i in range(k + 1)) * 2 / (2 ** n)
    return min(1.0, p), n10, n01

print("=" * 78)
print(f"RareBench retrieval ablation — {len(RB_RANKS)} cases")
print("=" * 78)
print(f"{'method':20s}{'R@1':>8s}{'R@5':>8s}{'R@10':>8s}{'R@25':>8s}"
      f"{'median':>9s}{'95% CI (R@25)':>22s}")
for m in METH:
    r = RB_RANKS[m].values
    lo, hi = boot_ci(r <= 25)
    print(f"{m:20s}{np.mean(r<=1):>8.3f}{np.mean(r<=5):>8.3f}"
          f"{np.mean(r<=10):>8.3f}{np.mean(r<=25):>8.3f}"
          f"{np.median(r):>9.0f}{f'[{lo:.3f}, {hi:.3f}]':>22s}")

print("\nSide by side with our own test set (R@25):")
print(f"{'method':20s}{'T0':>8s}{'T2':>8s}{'T3':>8s}{'RareBench':>12s}")
# recomputed from retrieval_results, never hardcoded
for m in METH:
    if (TIERS[0], m) not in retrieval_results:
        continue
    a, b, c_ = (retrieval_results[("T0", m)][25],
                retrieval_results[("T2", m)][25],
                retrieval_results[("T3", m)][25])
    print(f"{m:20s}{a:>8.2f}{b:>8.2f}{c_:>8.2f}"
          f"{np.mean(RB_RANKS[m].values<=25):>12.3f}")

print("\n" + "-" * 78)
print("Paired McNemar at R@25 (exact, two-sided)")
print("-" * 78)
pairs = [("hybrid_tuned", "bm25"), ("ic", "bm25"),
         ("ic", "hybrid_tuned"), ("hybrid_tuned", "hybrid"),
         ("hybrid", "bm25")]
pairs += [(b, "hybrid") for b in BASELINE_METHODS]
pairs += [(b, "ic") for b in BASELINE_METHODS]   # vs the closest in-house scorer
for a, b in pairs:
    ha = RB_RANKS[a].values <= 25
    hb = RB_RANKS[b].values <= 25
    p, n10, n01 = mcnemar_exact(ha, hb)
    reg_p("F3_external_retrieval", f"{a} vs {b} (R@25)", p)
    print(f"  {a:18s} vs {b:18s} {ha.mean():.3f} vs {hb.mean():.3f}  "
          f"delta={ha.mean()-hb.mean():+.3f}  b={n10:3d} c={n01:3d}  p={p:.4g}")

print("\n" + "-" * 78)
print("R@25 by subset")
print("-" * 78)
piv = RB_RANKS.groupby("subset").apply(
    lambda g: pd.Series({m: np.mean(g[m] <= 25) for m in METH}),
    include_groups=False)
piv.insert(0, "n", RB_RANKS.groupby("subset").size())
print(piv.round(3).to_string())

print("\nR@25 by query length (length is a confound vs our tiers)")
print(RB_RANKS.groupby("len_bin").apply(
    lambda g: pd.Series({"n": len(g),
                         **{m: np.mean(g[m] <= 25) for m in
                            ["bm25", "ic", "hybrid_tuned"]}}),
    include_groups=False).round(3).to_string())

print("\nSingle-gold cases only (n=%d), R@25:"
      % int(RB_RANKS.single_gold.sum()))
sg = RB_RANKS[RB_RANKS.single_gold]
print("  " + "  ".join(f"{m}={np.mean(sg[m]<=25):.3f}"
                       for m in ["bm25", "ic", "hybrid_tuned"]))

RB_RANKS.to_pickle(OUT + "rarebench_ranks.pkl")
piv.to_csv(OUT + "table_rarebench_retrieval.csv")
print("\nSaved rarebench_ranks.pkl + table_rarebench_retrieval.csv")

In [ ]:
# Patch v2. Cancel Run first (kernel stays alive, model stays loaded), then run
# this, then re-run R4 — it resumes from the checkpoint and re-pays for nothing.
from transformers import StoppingCriteria, StoppingCriteriaList

LOCAL_MAX_NEW_TOKENS = 140      # ranking output is ~110 tokens; 200 was slack
LOCAL_BATCH_STATE["size"] = 8   # auto-halves on OOM

class _AllClosed(StoppingCriteria):
    """Stop once every sequence has closed its JSON array or hit EOS."""
    def __init__(self, tok, n_in, batch):
        self.tok, self.n_in, self.done = tok, n_in, [False] * batch
    def __call__(self, input_ids, scores, **kw):
        for i in range(input_ids.shape[0]):
            if self.done[i]:
                continue
            gen = input_ids[i][self.n_in:]
            if self.tok.eos_token_id is not None and (gen == self.tok.eos_token_id).any():
                self.done[i] = True
            elif "]" in self.tok.decode(gen, skip_special_tokens=True):
                self.done[i] = True
        return all(self.done)

def _call_local_batch(model, prompts, max_new_tokens=None):
    tok, mdl = LOCAL[model]["tok"], LOCAL[model]["model"]
    mnt = max_new_tokens or LOCAL_MAX_NEW_TOKENS
    texts = []
    for sp, up in prompts:
        try:
            t = tok.apply_chat_template(
                [{"role": "system", "content": sp}, {"role": "user", "content": up}],
                add_generation_prompt=True, tokenize=False)
        except Exception:
            t = tok.apply_chat_template(
                [{"role": "user", "content": sp + "\n\n" + up}],
                add_generation_prompt=True, tokenize=False)
        texts.append(t)

    old_side = tok.padding_side
    tok.padding_side = "left"
    if tok.pad_token_id is None:
        tok.pad_token = tok.eos_token
    try:
        enc = tok(texts, return_tensors="pt", padding=True,
                  add_special_tokens=False).to(mdl.device)
        n_in = enc["input_ids"].shape[-1]
        with torch.no_grad():
            out = mdl.generate(
                **enc, max_new_tokens=mnt, do_sample=False, num_beams=1,
                pad_token_id=tok.pad_token_id,
                stopping_criteria=StoppingCriteriaList(
                    [_AllClosed(tok, n_in, len(texts))]))
    finally:
        tok.padding_side = old_side

    res = []
    for i in range(out.shape[0]):
        new = out[i][n_in:]
        res.append((tok.decode(new, skip_special_tokens=True),
                    int(enc["attention_mask"][i].sum()) + int(new.shape[-1])))
    return res

SESSION_BUDGET_H = 4.5   # ~5h45m of wall time left; stop with margin to spare
print("patched — re-run R4")

## R4. External LLM re-rank over ALL 834 cases

This replaces v4's 120-case stratified sample. It is resumable and stops cleanly at `SESSION_BUDGET_H`, so a multi-session run costs nothing but wall clock. **Do not score anything until the completeness audit at the bottom of this cell shows 0 remaining.**

In [ ]:
# =====================================================================
# R4. External LLM re-rank over ALL 834 mapped RareBench cases.
#     Resumable, batched, budget-aware. Re-run in a fresh session to
#     continue; it re-pays for nothing already stored.
# =====================================================================
RB_CKPT = OUT + "rarebench_rerank_v5.pkl"

rb_cases = pd.read_pickle(OUT + "rarebench_cases.pkl")
RB_RANKS = pd.read_pickle(OUT + "rarebench_ranks.pkl")

# Full set. No sampling. `single_gold` is kept as a reported subgroup, not a filter.
rb_run = rb_cases.reset_index(drop=True)
print(f"External workload: {len(rb_run)} cases x {len(ENSEMBLE_EXTERNAL)} models "
      f"= {len(rb_run)*len(ENSEMBLE_EXTERNAL)} calls")
print(f"retriever: {EXT_RETRIEVER}")


def rb_load():
    if os.path.exists(RB_CKPT):
        with open(RB_CKPT, "rb") as f:
            return pickle.load(f)
    return {}


def rb_save(d):
    tmp = RB_CKPT + ".tmp"
    with open(tmp, "wb") as f:
        pickle.dump(d, f)
    os.replace(tmp, RB_CKPT)


store = rb_load()
store.setdefault(("meta",), {})["retriever"] = EXT_RETRIEVER

# ---- candidates once, for every case (cheap, no LLM) -----------------
_new = 0
for _, row in rb_run.iterrows():
    k = ("cands", row["case_id"])
    if k not in store:
        idx, _ = retrieve(row, EXT_RETRIEVER, top_k=TOP_K_RETRIEVE)
        store[k] = [int(x) for x in idx]
        _new += 1
if _new:
    rb_save(store)
print(f"candidate lists: {_new} new, {len(rb_run)} total\n")

t_start = time.time()
budget_s = SESSION_BUDGET_H * 3600
stopped = False

for m in ENSEMBLE_EXTERNAL:
    todo = [i for i in range(len(rb_run))
            if (rb_run.iloc[i]["case_id"], m) not in store]
    print(f"=== {m.split('/')[-1]}: {len(todo)} remaining of {len(rb_run)} ===")
    if not todo:
        continue

    bs = LOCAL_BATCH if LIMITS[m]["provider"] == "local" else 8
    done_this = 0
    for s0 in range(0, len(todo), bs):
        if time.time() - t_start > budget_s:
            print(f"\n  [BUDGET] {SESSION_BUDGET_H} h reached. Stopping cleanly.")
            print( "  Everything so far is checkpointed. Re-run this cell in a new session.")
            stopped = True
            break

        chunk = todo[s0:s0 + bs]
        rows = [rb_run.iloc[i] for i in chunk]
        cands = [store[("cands", r["case_id"])] for r in rows]
        try:
            res = rerank_batch(m, rows, cands)
        except QuotaExhausted as e:
            print(f"\n  [QUOTA] {e}\n  -> skipping {m}; re-run tomorrow to resume.")
            break
        for r, rec in zip(rows, res):
            store[(r["case_id"], m)] = rec
        rb_save(store)
        done_this += len(chunk)

        if (s0 // bs) % 5 == 0 or s0 + bs >= len(todo):
            el = time.time() - t_start
            rate = done_this / max(1e-9, el)
            eta = (len(todo) - done_this) / rate / 60 if rate > 0 else float("nan")
            nerr = sum(1 for rec in res if rec["status"] == "error")
            print(f"  {done_this}/{len(todo)} | err_in_chunk={nerr} | "
                  f"elapsed {el/60:.0f}m | ETA {eta:.0f}m")
    if stopped:
        break

print(f"\nWall time this session: {(time.time()-t_start)/3600:.2f} h")
print(usage_report().to_string(index=False))

# ---- completeness audit ----------------------------------------------
print(f"\n{'model':34s}{'ok':>7s}{'err':>7s}{'todo':>7s}")
complete = True
for m in ENSEMBLE_EXTERNAL:
    ok = sum(1 for r in rb_run["case_id"]
             if isinstance(store.get((r, m)), dict)
             and store[(r, m)].get("status") == "ok")
    er = sum(1 for r in rb_run["case_id"]
             if isinstance(store.get((r, m)), dict)
             and store[(r, m)].get("status") == "error")
    todo = len(rb_run) - ok - er
    complete &= (todo == 0)
    print(f"{m:34s}{ok:>7d}{er:>7d}{todo:>7d}")

if not complete:
    print("\n>>> INCOMPLETE. Re-run this cell in a new session before scoring.")
    print(">>> Do NOT report partial coverage as a result — that is exactly the")
    print(">>> mistake v4 made with Gemini at 30/120.")
else:
    print("\nAll external calls complete.")


In [ ]:
# =====================================================================
# R5. External analysis on the full mapped set. Conditional analysis is
#     PRIMARY (it measures re-ranking); unconditional is reported too
#     (it is bounded by retrieval and is what an end-user would see).
# =====================================================================
from scipy.stats import kendalltau

with open(RB_CKPT, "rb") as f:
    store = pickle.load(f)
rb_cases = pd.read_pickle(OUT + "rarebench_cases.pkl")

SHORT = {"google/medgemma-27b-text-it": "medgemma",
         "mistral-small-latest": "mistral",
         "gemini-3.5-flash-lite": "gemini"}
PAIR = [m for m in ENSEMBLE_EXTERNAL]


def gold_rank(ranking, gold):
    for r, item in enumerate(ranking):
        if int(item) in gold:
            return r + 1
    return None


rows, n_skip = [], 0
for _, c in rb_cases.iterrows():
    cid, gold = c["case_id"], set(int(x) for x in c["gold_kb_idx"])
    cand = store.get(("cands", cid))
    if cand is None:
        n_skip += 1
        continue
    got = {}
    for m in PAIR:
        v = store.get((cid, m))
        if isinstance(v, dict) and v.get("status") == "ok":
            got[m] = v["ranking"]
    if not all(m in got for m in PAIR):
        n_skip += 1
        continue

    rec = dict(case_id=cid, subset=c["subset"], n_phe=c["n_phe"],
               len_bin=c["len_bin"], single_gold=c["single_gold"],
               retr_rank=gold_rank(cand, gold))
    rec["retrievable"] = rec["retr_rank"] is not None
    for m in PAIR:
        rec[f"rank_{SHORT[m]}"] = gold_rank(got[m], gold)
    rec["rank_ensemble"] = gold_rank(
        rrf_fuse_lists([got[m] for m in PAIR], k=60), gold)
    _rr = [rec[f"rank_{SHORT[m]}"] for m in PAIR
           if rec[f"rank_{SHORT[m]}"] is not None]
    rec["rank_oracle"] = min(_rr) if _rr else None

    if len(PAIR) >= 2:
        p1 = {int(it): r for r, it in enumerate(got[PAIR[0]])}
        p2 = {int(it): r for r, it in enumerate(got[PAIR[1]])}
        keys = sorted(set(p1) & set(p2))
        tau, _ = kendalltau([p1[k] for k in keys], [p2[k] for k in keys]) \
            if len(keys) >= 3 else (np.nan, None)
        rec["tau"] = tau
        rec["disagreement"] = (1 - tau) / 2 if tau == tau else np.nan
    rows.append(rec)

R = pd.DataFrame(rows)
print(f"Analysed {len(R)} cases ({n_skip} skipped for incomplete model coverage)")
if n_skip > 0.02 * len(rb_cases):
    print(f"[!] {n_skip} skipped is more than 2% — investigate before reporting.")
print(f"gold in top-{TOP_K_RETRIEVE}: {int(R.retrievable.sum())} "
      f"({R.retrievable.mean():.1%})")

COLS = ["retr"] + [SHORT[m] for m in PAIR] + ["ensemble", "oracle"]
_col = lambda n: "retr_rank" if n == "retr" else f"rank_{n}"
_hits = lambda df, n, k: np.array(
    [(x is not None) and (x == x) and (x <= k) for x in df[_col(n)]])
_mrr = lambda df, n: float(np.mean(
    [1.0 / x if (x is not None and x == x) else 0.0 for x in df[_col(n)]]))

Rr = R[R.retrievable]
for label, D in [(f"CONDITIONAL on gold retrievable (n={len(Rr)}) — PRIMARY, "
                  f"measures RE-RANKING", Rr),
                 (f"UNCONDITIONAL (all {len(R)}) — bounded by retrieval", R)]:
    print("\n" + "=" * 74); print(label); print("=" * 74)
    print(f"{'system':14s}{'R@1':>9s}{'R@5':>9s}{'R@10':>9s}{'MRR':>9s}")
    for nm in COLS:
        print(f"{nm:14s}{_hits(D,nm,1).mean():>9.3f}{_hits(D,nm,5).mean():>9.3f}"
              f"{_hits(D,nm,10).mean():>9.3f}{_mrr(D,nm):>9.3f}")

print("\n" + "-" * 74)
print("McNemar (exact, two-sided), conditional set, R@5 — family F4_external_llm")
print("-" * 74)
_pairs = [(s, "retr") for s in [SHORT[m] for m in PAIR] + ["ensemble"]]
_pairs += [("ensemble", SHORT[PAIR[0]])]
for a, b in _pairs:
    ha, hb = _hits(Rr, a, 5), _hits(Rr, b, 5)
    p, n10, n01 = mcnemar_exact(ha, hb)
    reg_p("F4_external_llm", f"{a} vs {b} (R@5, conditional)", p,
          a=ha.mean(), b=hb.mean(), n=len(Rr))
    print(f"  {a:10s} vs {b:10s} {ha.mean():.3f} vs {hb.mean():.3f}"
          f"  b={n10:3d} c={n01:3d}  p_raw={p:.4g}")
print("  (corrected p-values appear in the multiplicity cell — quote those)")

print("\n" + "-" * 74); print("Per-subset R@5 (conditional)"); print("-" * 74)
print(Rr.groupby("subset").apply(
    lambda g: pd.Series({"n": len(g), **{nm: _hits(g, nm, 5).mean() for nm in COLS}}),
    include_groups=False).round(3).to_string())

print("\n" + "-" * 74); print("Single-gold cases only (conditional)"); print("-" * 74)
_sg = Rr[Rr.single_gold]
print(f"  n={len(_sg)}  " + "  ".join(
    f"{nm}={_hits(_sg,nm,5).mean():.3f}" for nm in COLS))

if "disagreement" in R and R.disagreement.notna().any():
    print("\n" + "-" * 74)
    print(f"Selective prediction — disagreement between "
          f"{SHORT[PAIR[0]]} and {SHORT[PAIR[1]]}")
    print("-" * 74)
    d = R.dropna(subset=["disagreement"]).sort_values("disagreement")
    h1 = _hits(d, "ensemble", 1).astype(float)
    cov = np.arange(1, len(d) + 1) / len(d)
    acc = np.cumsum(h1) / np.arange(1, len(d) + 1)
    aurc = float(np.trapezoid(acc, cov))
    rs = np.random.RandomState(0)
    rand = float(np.mean([np.trapezoid(
        np.cumsum(h1[rs.permutation(len(h1))]) / np.arange(1, len(h1) + 1), cov)
        for _ in range(50)]))
    print(f"  mean tau = {R.tau.mean():.3f}  sd = {R.tau.std():.3f}")
    print(f"  AURC(accuracy-coverage) {aurc:.4f} vs random {rand:.4f}"
          f"  -> {'helps' if aurc > rand else 'NO selective benefit'}")

R.to_pickle(OUT + "rarebench_analysis_v5.pkl")
R.to_csv(OUT + "table_rarebench_llm_v5.csv", index=False)
print("\nSaved rarebench_analysis_v5.pkl + table_rarebench_llm_v5.csv")


## R8. RareBench's published numbers, with the non-comparability block

In [ ]:
# =====================================================================
# R8. RareBench's own published results, for context.
#
#     SOURCE (verified by reading the paper, not from memory):
#     Chen, Mao, Guo, Wang, Zhang & Chen. "RareBench: Can LLMs Serve as Rare
#     Diseases Specialists?" KDD '24. arXiv:2402.06341. Table 4, Task 4
#     (Differential Diagnosis among Universal Rare Diseases).
#
#     ############   READ THIS OR YOU WILL WRITE A FALSE CLAIM   ############
#     These numbers are NOT directly comparable to ours. Four reasons, all of
#     which must appear in the paper next to the table:
#
#       1. DIFFERENT CASES. Theirs are 2,185 cases = the 1,122 public cases PLUS
#          PUMCH hospital records. Ours are the 834 public cases that carry an
#          ORPHA code present in our Orphanet KB. Different denominator, and ours
#          is a filtered subset that drops 288 cases (LIRICAL keeps only 53%).
#       2. DIFFERENT TASK. RareBench Task 4 is open-ended generation of a top-10
#          differential over all known rare diseases. Ours is re-ranking a
#          retrieved list of 25 candidates drawn from 4,289 Orphanet disorders.
#          Re-ranking a shortlist is an easier task than open generation.
#       3. DIFFERENT LABEL SPACE. Their knowledge graph spans 9,260 disease
#          entities from HPO/OMIM/Orphanet/CCRD; ours spans 4,289 Orphanet rows.
#       4. DIFFERENT ERA. GPT-4-1106-preview, evaluated in 2024.
#
#     Legitimate use: "for reference, on the full RareBench Task 4 set GPT-4
#     zero-shot reaches hit@1 0.323". Illegitimate: "our system beats GPT-4".
#     #######################################################################
# =====================================================================
RAREBENCH_TASK4 = pd.DataFrame([
    # model,                setting,   hit@1, hit@3, hit@10, median rank
    ("GPT-4",               "0-shot",   32.3,  45.4,  58.9,  "5.0"),
    ("GPT-4",               "3-shot",   30.4,  43.8,  57.9,  "5.0"),
    ("GPT-4",               "CoT",      33.2,  46.4,  59.5,  "4.0"),
    ("GPT-3.5-Turbo",       "0-shot",   21.1,  34.2,  48.2,  ">10"),
    ("Gemini Pro",          "0-shot",   14.6,  22.7,  33.0,  ">10"),
    ("GLM4",                "0-shot",   19.1,  30.4,  45.5,  ">10"),
    ("GLM3-Turbo",          "0-shot",   12.4,  21.0,  33.2,  ">10"),
    ("Mistral-7B",          "0-shot",    7.2,  12.5,  18.9,  ">10"),
    ("Llama2-7B",           "0-shot",    7.4,  11.0,  14.8,  ">10"),
    ("ChatGLM3-6B",         "0-shot",    5.0,   7.2,  10.7,  ">10"),
    ("BioMistral-7B",       "0-shot",    6.5,   8.5,  12.3,  ">10"),
    ("HuatuoGPT2-7B",       "0-shot",   11.4,  17.8,  28.1,  ">10"),
    ("MedAlpaca-7B",        "0-shot",    8.4,  14.3,  19.4,  ">10"),
], columns=["model", "setting", "hit@1", "hit@3", "hit@10", "median_rank"])
RAREBENCH_TASK4["n_cases"] = 2185
RAREBENCH_TASK4["source"] = "Chen et al. KDD'24, Table 4 (Task 4)"

print("RareBench Task 4, published (n=2,185 cases, 421 diseases):")
print(RAREBENCH_TASK4[["model", "setting", "hit@1", "hit@3", "hit@10",
                       "median_rank"]].to_string(index=False))

R = pd.read_pickle(OUT + "rarebench_analysis_v5.pkl")
Rr = R[R.retrievable]
_h = lambda df, c, k: float(np.mean([(x is not None and x == x and x <= k)
                                     for x in df[c]]))
print(f"\nOurs, for reference only (n={len(R)} public cases, ORPHA-codeable, "
      f"re-ranking a 25-candidate shortlist):")
print(f"{'system':22s}{'hit@1':>8s}{'hit@3':>8s}{'hit@10':>8s}")
for nm, col in [("retrieval only", "retr_rank"),
                ("MedGemma re-rank", "rank_medgemma"),
                ("ensemble", "rank_ensemble")]:
    if col in R:
        print(f"{nm:22s}{_h(R,col,1)*100:>8.1f}{_h(R,col,3)*100:>8.1f}"
              f"{_h(R,col,10)*100:>8.1f}")

print("\n>>> Put the four non-comparability reasons in the table caption, not in")
print(">>> a footnote. A reviewer who spots an implied head-to-head will assume")
print(">>> the worst about the rest of the paper.")

RAREBENCH_TASK4.to_csv(OUT + "table_rarebench_published.csv", index=False)


# Part R9-R10 — the paper's tables and figures

In [ ]:
# =====================================================================
# R9. The paper's headline tables. Everything is RECOMPUTED from stored
#     artefacts — no hardcoded constants copied from an earlier print.
# =====================================================================
R  = pd.read_pickle(OUT + "rarebench_analysis_v5.pkl")
Rr = R[R.retrievable]
RB_RANKS = pd.read_pickle(OUT + "rarebench_ranks.pkl")
istore = load_ckpt()

_h  = lambda df, c, k: float(np.mean([(x is not None and x == x and x <= k)
                                      for x in df[c]]))

def boot_ci(h, B=2000, seed=SEED):
    h = np.asarray(h, float); rs = np.random.RandomState(seed)
    return tuple(np.percentile(
        [h[rs.randint(0, len(h), len(h))].mean() for _ in range(B)], [2.5, 97.5]))

# ---- internal LLM numbers, recomputed from the internal checkpoint ----
def internal_r5(tier, models):
    df = test_sets[tier]
    got_r, got_l, got_e = [], [], []
    for ci in range(len(df)):
        cands = istore.get(("cands", tier, ci))
        recs = [istore.get((tier, ci, m)) for m in models]
        if cands is None or any(r is None or r.get("status") != "ok" for r in recs):
            continue
        true = df.iloc[ci]["true_disease"]
        hit = lambda rk: true in kb.iloc[rk[:5]]["disease_name"].tolist()
        got_r.append(hit(cands))
        got_l.append([hit(r["ranking"]) for r in recs])
        got_e.append(hit(rrf_fuse_lists([r["ranking"] for r in recs], k=60)))
    if not got_r:
        return None
    L = np.array(got_l)
    return dict(n=len(got_r), retr=float(np.mean(got_r)),
                per_model=L.mean(axis=0), best_llm=float(L.mean(axis=0).max()),
                ens=float(np.mean(got_e)))

INT = {t: internal_r5(t, ENSEMBLE_INTERNAL) for t in TIERS}

print("=" * 80)
print("TABLE 1 — retrieval R@25: internal (synthetic) vs external (RareBench)")
print("=" * 80)
_int_r25 = {m: {t: retrieval_results[(t, m)][TOP_K_RETRIEVE]
                for t in TIERS if (t, m) in retrieval_results}
            for m in METHODS_V5}
print(f"{'method':20s}" + "".join(f"{t:>7s}" for t in TIERS)
      + f"{'RareBench':>11s}{'95% CI':>18s}")
t1 = []
for m in METHODS_V5:
    if m not in RB_RANKS.columns:
        continue
    hits = RB_RANKS[m].values <= TOP_K_RETRIEVE
    lo, hi = boot_ci(hits)
    row = {"method": m, **{t: _int_r25.get(m, {}).get(t, np.nan) for t in TIERS},
           "rarebench": hits.mean(), "ci_lo": lo, "ci_hi": hi}
    t1.append(row)
    print(f"{m:20s}" + "".join(f"{row[t]:>7.2f}" if row[t] == row[t] else f"{'-':>7s}"
                               for t in TIERS)
          + f"{hits.mean():>11.3f}{f'[{lo:.3f},{hi:.3f}]':>18s}")
pd.DataFrame(t1).to_csv(OUT + "table1_retrieval.csv", index=False)

print("\n" + "=" * 80)
print("TABLE 2 — THE INVERSION: LLM re-rank vs its own retrieval baseline, R@5")
print("=" * 80)
print(f"{'condition':26s}{'n':>6s}{'retrieval':>11s}{'best LLM':>10s}"
      f"{'delta':>9s}{'ensemble':>10s}")
t2 = []
for t in TIERS:
    d = INT[t]
    if d is None:
        continue
    t2.append(dict(condition=f"internal {t}", n=d["n"], retrieval=d["retr"],
                   best_llm=d["best_llm"], delta=d["best_llm"] - d["retr"],
                   ensemble=d["ens"]))
    print(f"{'internal '+t:26s}{d['n']:>6d}{d['retr']:>11.3f}{d['best_llm']:>10.3f}"
          f"{d['best_llm']-d['retr']:>+9.3f}{d['ens']:>10.3f}")

_llm_cols = [f"rank_{SHORT[m]}" for m in ENSEMBLE_EXTERNAL if f"rank_{SHORT[m]}" in R]
for lbl, D in [("RareBench conditional", Rr), ("RareBench unconditional", R)]:
    r_ = _h(D, "retr_rank", 5)
    b_ = max(_h(D, c, 5) for c in _llm_cols)
    e_ = _h(D, "rank_ensemble", 5)
    t2.append(dict(condition=lbl, n=len(D), retrieval=r_, best_llm=b_,
                   delta=b_ - r_, ensemble=e_))
    print(f"{lbl:26s}{len(D):>6d}{r_:>11.3f}{b_:>10.3f}{b_-r_:>+9.3f}{e_:>10.3f}")
pd.DataFrame(t2).to_csv(OUT + "table2_inversion.csv", index=False)
print("\n  The delta is negative on every synthetic tier and positive externally.")
print("  That sign change is the paper.")

print("\n" + "=" * 80)
print("TABLE 3 — TUNING TRANSFER: dev-tuned fusion vs untuned, internal vs external")
print("=" * 80)
print(f"{'retriever':20s}{'internal mean R@25':>20s}{'RareBench R@25':>17s}{'rank':>7s}")
_ext = {m: (RB_RANKS[m].values <= TOP_K_RETRIEVE).mean()
        for m in METHODS_V5 if m in RB_RANKS.columns}
_ord = sorted(_ext, key=lambda m: -_ext[m])
t3 = []
for m in ["hybrid_tuned", "hybrid", "ic", "bm25", "sapbert"] + BASELINE_METHODS:
    if m not in _ext:
        continue
    im = np.mean([_int_r25[m][t] for t in TIERS]) if m in _int_r25 and _int_r25[m] else np.nan
    t3.append(dict(retriever=m, internal_mean_r25=im, rarebench_r25=_ext[m],
                   external_rank=_ord.index(m) + 1))
    print(f"{m:20s}{im:>20.3f}{_ext[m]:>17.3f}{_ord.index(m)+1:>7d}")
pd.DataFrame(t3).to_csv(OUT + "table3_tuning_transfer.csv", index=False)
print(f"\n  hybrid_tuned is the best internal retriever and ranks "
      f"{_ord.index('hybrid_tuned')+1} of {len(_ord)} externally.")
print("  Weights fitted on same-source synthetic data do not merely fail to")
print("  transfer — they transfer negatively.")

print("\n" + "=" * 80)
print("TABLE 4 — ensemble vs best single model (the negative result)")
print("=" * 80)
for lbl, D in [("RareBench conditional", Rr), ("RareBench unconditional", R)]:
    best_c = max(_llm_cols, key=lambda c: _h(D, c, 5))
    a, b = _h(D, "rank_ensemble", 5), _h(D, best_c, 5)
    ha = np.array([(x is not None and x == x and x <= 5) for x in D["rank_ensemble"]])
    hb = np.array([(x is not None and x == x and x <= 5) for x in D[best_c]])
    p, n10, n01 = mcnemar_exact(ha, hb)
    print(f"  {lbl:26s} ensemble={a:.3f}  best({best_c[5:]})={b:.3f}  "
          f"delta={a-b:+.3f}  p_raw={p:.4g}")
print("\n  Report this as: fusion gave no benefit over the best single member at")
print("  this sample size. Do not present the ensemble as a contribution.")


In [ ]:
# =====================================================================
# R10. Figure 1 — the inversion. Numbers come from table2_inversion.csv,
#      so the figure can never drift from the table.
# =====================================================================
import matplotlib.pyplot as plt

t2 = pd.read_csv(OUT + "table2_inversion.csv")
t2p = t2[t2.condition.isin([f"internal {t}" for t in TIERS]
                           + ["RareBench unconditional"])].reset_index(drop=True)
lbl = [c.replace("internal ", "").replace("RareBench unconditional", "RareBench")
       for c in t2p.condition]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11.5, 4.3))
x, w = np.arange(len(t2p)), 0.27
ax1.bar(x - w, t2p.retrieval, w, label="Retrieval", color="#4C72B0")
ax1.bar(x,     t2p.best_llm,  w, label="Best single LLM", color="#DD8452")
ax1.bar(x + w, t2p.ensemble,  w, label="Ensemble (RRF)", color="#55A868")
ax1.axvline(len(t2p) - 1.5, color="k", ls=":", lw=1.2)
ax1.text(len(t2p) - 1.45, 0.96, "external", fontsize=8, style="italic")
ax1.set_xticks(x); ax1.set_xticklabels(lbl)
ax1.set_ylabel("Recall@5"); ax1.set_ylim(0, 1.05)
ax1.set_title("(a) LLM re-ranking vs its retrieval baseline")
ax1.legend(fontsize=8, loc="upper right"); ax1.grid(axis="y", alpha=.3)

ax2.axhline(0, color="k", lw=1)
ax2.scatter(t2p.retrieval[:-1], t2p.delta[:-1], s=90, color="#4C72B0",
            zorder=3, label="internal (synthetic queries)")
ax2.scatter(t2p.retrieval.iloc[-1:], t2p.delta.iloc[-1:], s=140, marker="D",
            color="#C44E52", zorder=3, label="external (real cases)")
for xx, yy, nn in zip(t2p.retrieval, t2p.delta, lbl):
    ax2.annotate(nn, (xx, yy), textcoords="offset points", xytext=(6, 5), fontsize=8)
ax2.set_xlabel("Retrieval baseline Recall@5")
ax2.set_ylabel("LLM re-rank minus retrieval")
ax2.set_title("(b) The penalty is an artefact of the baseline")
ax2.legend(fontsize=8); ax2.grid(alpha=.3)

plt.tight_layout()
for ext in ("png", "pdf"):
    plt.savefig(OUT + f"fig1_inversion.{ext}", dpi=300, bbox_inches="tight")
plt.show()

# ---- Figure 2: tuning transfer ----------------------------------------
t3 = pd.read_csv(OUT + "table3_tuning_transfer.csv").dropna(subset=["internal_mean_r25"])
fig, ax = plt.subplots(figsize=(6.2, 4.6))
ax.scatter(t3.internal_mean_r25, t3.rarebench_r25, s=80, color="#4C72B0", zorder=3)
for _, r in t3.iterrows():
    col = "#C44E52" if r.retriever == "hybrid_tuned" else "#333333"
    ax.annotate(r.retriever, (r.internal_mean_r25, r.rarebench_r25),
                textcoords="offset points", xytext=(6, 4), fontsize=8, color=col)
lim = [0, 1.02]
ax.plot(lim, lim, ls=":", color="grey", lw=1)
ax.set_xlim(lim); ax.set_ylim(0, 0.8)
ax.set_xlabel("Internal mean Recall@25 (synthetic tiers)")
ax.set_ylabel("RareBench Recall@25")
ax.set_title("Internal retrieval performance does not predict external")
ax.grid(alpha=.3)
plt.tight_layout()
for ext in ("png", "pdf"):
    plt.savefig(OUT + f"fig2_tuning_transfer.{ext}", dpi=300, bbox_inches="tight")
plt.show()
print("Saved fig1_inversion and fig2_tuning_transfer (png + pdf)")


# Statistics: multiplicity correction

In [ ]:
# =====================================================================
# S1. Multiplicity correction. Holm (FWER) and Benjamini-Hochberg (FDR)
#     within each pre-declared family. QUOTE THE CORRECTED VALUES.
# =====================================================================
pv = pval_table()
if pv.empty:
    print("[!] No p-values registered. Run the cells that call reg_p() first.")
else:
    pv = pv.sort_values(["family", "p_raw"]).reset_index(drop=True)
    for fam, g in pv.groupby("family"):
        print("=" * 84)
        print(f"{fam}   ({FAMILIES.get(fam,'?')})   m = {len(g)} tests")
        print("=" * 84)
        print(f"{'test':46s}{'p_raw':>10s}{'p_holm':>10s}{'p_BH':>10s}{'sig':>7s}")
        for _, r in g.iterrows():
            star = "*" if r.get("sig_holm") else ""
            print(f"{r['label'][:45]:46s}{r['p_raw']:>10.4g}"
                  f"{r['p_holm']:>10.4g}{r['p_fdr_bh']:>10.4g}{star:>7s}")
        print()

    n_raw  = int((pv.p_raw < ALPHA).sum())
    n_holm = int(pv.sig_holm.sum())
    n_bh   = int(pv.sig_fdr_bh.sum())
    print(f"Significant at alpha={ALPHA}:  raw {n_raw}/{len(pv)}   "
          f"Holm {n_holm}/{len(pv)}   BH {n_bh}/{len(pv)}")
    if n_raw > n_holm:
        print(f"\n>>> {n_raw-n_holm} test(s) lose significance under Holm. Those")
        print(">>> claims must be softened or dropped from the paper.")
    pv.to_csv(OUT + "table_pvalues_corrected.csv", index=False)
    print("\nSaved table_pvalues_corrected.csv")


In [ ]:
# =====================================================================
# Z. Provenance. Paste this into the Methods section verbatim.
# =====================================================================
import platform, transformers, sentence_transformers
prov = {
    "seed": SEED,
    "python": sys.version.split()[0],
    "platform": platform.platform(),
    "torch": torch.__version__,
    "transformers": transformers.__version__,
    "sentence_transformers": sentence_transformers.__version__,
    "hpo_source_url": HPO_URL,
    "hpo_data_version": HPO_DATA_VERSION,
    "kb_n_diseases": int(len(kb)),
    "medgemma_source": LOCAL.get(MEDGEMMA_ID, {}).get("source"),
    "medgemma_quant": "4-bit NF4, double quant",
    "medgemma_compute_dtype": globals().get("MEDGEMMA_COMPUTE_DTYPE", "unknown"),
    "ensemble_internal": ENSEMBLE_INTERNAL,
    "ensemble_external": ENSEMBLE_EXTERNAL,
    "ext_retriever": EXT_RETRIEVER,
    "ext_retriever_choice": "POST-HOC, externally informed — declare in the paper",
    "best_rrf": BEST_RRF,
    "top_k_retrieve": TOP_K_RETRIEVE,
    "local_batch_final": LOCAL_BATCH_STATE.get("size"),
    "local_max_new_tokens": LOCAL_MAX_NEW_TOKENS,
    "alpha": ALPHA,
    "corrections": list(CORRECTIONS),
    "baselines_are_reimplementations": True,
    "anc_discount_lirical_style": ANC_DISCOUNT,
}
with open(OUT + "provenance_v5.json", "w") as f:
    json.dump(prov, f, indent=2)
print(json.dumps(prov, indent=2))

WANT = ["table1_retrieval.csv", "table2_inversion.csv", "table3_tuning_transfer.csv",
        "table_rarebench_retrieval.csv", "table_rarebench_llm_v5.csv",
        "table_rarebench_published.csv", "table_pvalues_corrected.csv",
        "table_rarebench_coverage.csv", "fig1_inversion.pdf",
        "fig2_tuning_transfer.pdf", "provenance_v5.json"]
print("\nDeliverables:")
for w in WANT:
    print(f"  {'OK ' if os.path.exists(OUT+w) else 'MISS'} {w}")
